In [1]:
# ============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================================
!pip install -q xgboost optuna shap scikit-learn pandas numpy matplotlib seaborn scipy openpyxl xlrd

import pandas as pd
import numpy as np
import warnings, json, pickle
from pathlib import Path
from datetime import datetime, timedelta
from typing import List # Add this import
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import xml.etree.ElementTree as ET

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ── CONFIGURATION CHEMINS ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive/MyDrive')
TERRAIN_ROOT    = DRIVE_ROOT / 'Données Météo'
ERA5_PATH       = DRIVE_ROOT / 'InteractiveSheet_2026-05-06_15_23_36.xlsx'
OPENMETEO_PATH  = DRIVE_ROOT / 'extraction_open_meteo_hourly_20260501_160120.csv'
NASA_PATH       = DRIVE_ROOT / 'extraction_nasa_power_20260423_164719.csv'
PROCESSED       = DRIVE_ROOT / 'data_processed_v3'
PROCESSED.mkdir(exist_ok=True)

# Plage de prédiction finale (modifier selon besoin)
PRED_DATE_START = '2023-01-01'
PRED_DATE_END   = '2025-12-31'

print("✅ Configuration OK")
print(f"   Terrain : {TERRAIN_ROOT}")
print(f"   Output  : {PROCESSED}")
print(f"   Prédiction de {PRED_DATE_START} à {PRED_DATE_END}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Configuration OK
   Terrain : /content/drive/MyDrive/MyDrive/Données Météo
   Output  : /content/drive/MyDrive/MyDrive/data_processed_v3
   Prédiction de 2023-01-01 à 2025-12-31


In [2]:
import os

# List contents of TERRAIN_ROOT
terrain_root_contents = os.listdir(TERRAIN_ROOT)

if terrain_root_contents:
    print(f"Files and directories found in {TERRAIN_ROOT}:")
    for item in terrain_root_contents:
        print(f"- {item}")
else:
    print(f"The directory {TERRAIN_ROOT} is empty.")

Files and directories found in /content/drive/MyDrive/MyDrive/Données Météo:
- 00208023 • DK-GHARBIA 343
- 00208068 • DK-USINE SB
- 0020807E • DK-USINE ZEMAMRA
- 002054AB • SUTA 503
- 0020C510 • SIDI SLIMANE
- 00208035 • DK-SEMVAD
- 002054AC • SUTA 505
- 00206F29 • M-Nador
- 002054AA • SUTA INRA
- 00206F18 • M-Berkane
- 002045CF • SUTA_CENTAGRI
- 002054CA • SUTA Ait Ali
- 002054CC • SUTA 522
- 002045CE • Merksen
- 0020804A • DK-FAREGH 310
- 00203F8F • M-Zaio
- 00001A46 • DAR ELGUEDDARI
- 00203F16 • MECHRAA BELKSIRI
- 00001A3F • ksar El Kebir
- 00203734 • SUTA OULAD AYAD
- 00203F1C • SUTA 507
- 00203733 • SUTA-TAZEROUALT
- 00002CE0 • SIDI ALLAL TAZI
- 00203662 • SUTA - FERME ABT


In [3]:
# ============================================================
# CELL 2 - PIPELINE TERRAIN ROBUSTE
# ============================================================

!pip install -q openpyxl xlrd lxml pandas -U

import pandas as pd
import numpy as np
from pathlib import Path
import xml.etree.ElementTree as ET
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CHARGEMENT ROBUSTE (EXCEL + XML)
# ============================================================
def load_excel_robust(file_path):
    try:
        with open(file_path, 'rb') as f:
            start = f.read(500)
    except:
        return None, None

    # XML
    if b'<?xml' in start:
        try:
            tree = ET.parse(file_path)
            root = tree.getroot()
            ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}
            rows = []
            for row in root.findall('.//ss:Row', ns):
                row_data = []
                for cell in row.findall('ss:Cell', ns):
                    data = cell.find('ss:Data', ns)
                    row_data.append(data.text if data is not None else None)
                rows.append(row_data)
            if not rows:
                return None, None
            df = pd.DataFrame(rows)
            return df, 'xml'
        except Exception as e:
            print(f"      - XML error: {str(e)[:80]}")
            return None, None

    # Excel normal
    try:
        df = pd.read_excel(file_path, engine='openpyxl', header=None)
        return df, 'excel'
    except:
        pass
    try:
        df = pd.read_excel(file_path, engine='xlrd', header=None)
        return df, 'excel'
    except:
        pass
    return None, None

# ============================================================
# 2. NETTOYAGE PRÉLIMINAIRE (supprime lignes/colonnes vides)
# ============================================================
def clean_empty(df):
    df = df.dropna(how='all').dropna(axis=1, how='all').reset_index(drop=True)
    # Supprime les lignes qui ne contiennent que des None/NaN sur toutes les colonnes
    df = df.dropna(how='all')
    return df

# ============================================================
# 3. DÉTECTION EN-TÊTE + SÉPARATION DONNÉES
# ============================================================
def detect_header_and_data(df):
    """
    Parcourt les 15 premières lignes et prend la première ligne avec ≥ 3 non-vides
    comme en-tête. Retourne (header_row_values, data_df).
    """
    if len(df) == 0:
        return None, None
    for i in range(min(15, len(df))):
        row = df.iloc[i]
        if row.notna().sum() >= 3:   # seuil : au moins 3 colonnes non vides
            header = row.values
            data = df.iloc[i+1:].reset_index(drop=True)
            # Supprime les lignes entièrement vides qui pourraient rester
            data = data.dropna(how='all').reset_index(drop=True)
            return header, data
    return None, None

# ============================================================
# 4. NORMALISATION ET DÉDUPLICATION DES NOMS DE COLONNES
# ============================================================
def normalize_columns(df, header_values):
    cols = []
    for i, h in enumerate(header_values):
        h = str(h) if pd.notna(h) else f"col_{i}"
        # nettoyage
        h = h.lower()
        for a, b in [(' ','_'),('°',''),('(',''),(')',''),('/','_'),('-','_'),('[',''),(']',''),('\n',''),('\r',''), ('ç','c')]:
            h = h.replace(a, b)
        cols.append(h)

    # Rendre unique : col, col_1, col_2 ...
    seen = {}
    unique_cols = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            unique_cols.append(c)
        else:
            seen[c] += 1
            unique_cols.append(f"{c}_{seen[c]}")
    df.columns = unique_cols
    return df

# ============================================================
# 5. DÉTECTION COLONNE DATE
# ============================================================
def detect_date_column(df):
    for col in df.columns:
        sample = df[col].dropna().astype(str).head(20)
        if len(sample) == 0:
            continue
        parsed = pd.to_datetime(sample, errors='coerce', dayfirst=True)
        if parsed.notna().sum() >= max(3, len(sample)*0.5):
            return col
    return None

# ============================================================
# 6. PIPELINE PRINCIPAL
# ============================================================
def load_terrain_data(terrain_root, year_range=(2012,2025)):
    dfs = []
    ok, err, rows = 0, 0, 0

    print("="*100)
    print(f"📁 Dossier: {terrain_root}")
    print("="*100)

    # Lister les fichiers
    files = []
    for region in sorted(terrain_root.iterdir()):
        if not region.is_dir():
            continue
        for f in region.glob("*.xls"):
            try:
                if year_range[0] <= int(f.stem) <= year_range[1]:
                    files.append((region.name, f))
            except:
                pass

    print(f"\n📄 Total fichiers: {len(files)}\n")
    print("="*100)

    for i, (region, file) in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {region:<30} | {file.name:<10}", end=" ")

        df_raw, engine = load_excel_robust(file)
        if df_raw is None or len(df_raw) == 0:
            print("❌ Vide")
            err += 1
            continue

        # Nettoyage des lignes/colonnes vides
        df_raw = clean_empty(df_raw)

        # Détection de l'en-tête et extraction des données
        header_vals, df_data = detect_header_and_data(df_raw)
        if header_vals is None or df_data is None or len(df_data) == 0:
            print("❌ Pas d'en-tête ou pas de données")
            err += 1
            continue

        # Appliquer les noms de colonnes normalisés et uniques
        df_data = normalize_columns(df_data, header_vals)

        # Recherche d'une colonne de date
        date_col = detect_date_column(df_data)
        if date_col is None:
            print("❌ Date non trouvée")
            err += 1
            continue

        # Conversion datetime
        df_data["datetime"] = pd.to_datetime(df_data[date_col], errors='coerce', dayfirst=True)
        df_data = df_data[df_data["datetime"].notna()]
        if len(df_data) == 0:
            print("❌ Aucune date valide")
            err += 1
            continue

        df_data["region_id"] = region
        df_data["source_file"] = file.name

        dfs.append(df_data)
        ok += 1
        rows += len(df_data)
        print(f"✅ {len(df_data):,} rows [{engine}]")

    print("\n"+"="*100)
    print(f"📊 OK: {ok} | ❌: {err} | ROWS: {rows:,}")
    print("="*100)

    if not dfs:
        return pd.DataFrame()

    # Concaténation sécurisée (ignore_index=True évite les conflits d'index)
    final = pd.concat(dfs, ignore_index=True)
    final = final.sort_values(["region_id", "datetime"]).reset_index(drop=True)

    print("✅ FINAL SHAPE:", final.shape)
    return final

# ============================================================
# 7. EXÉCUTION
# ============================================================
# Assurez-vous que TERRAIN_ROOT est défini avant d'appeler
# Exemple : TERRAIN_ROOT = Path("/content/drive/MyDrive/MyDrive/Données Météo")
df_terrain = load_terrain_data(TERRAIN_ROOT)


📁 Dossier: /content/drive/MyDrive/MyDrive/Données Météo

📄 Total fichiers: 197

[1/197] 00001A3F • ksar El Kebir       | 2021.xls   ✅ 7 rows [xml]
[2/197] 00001A3F • ksar El Kebir       | 2014.xls   ✅ 8,721 rows [xml]
[3/197] 00001A3F • ksar El Kebir       | 2020.xls   ✅ 997 rows [xml]
[4/197] 00001A3F • ksar El Kebir       | 2013.xls   ✅ 7,859 rows [xml]
[5/197] 00001A3F • ksar El Kebir       | 2015.xls   ✅ 6,016 rows [xml]
[6/197] 00001A3F • ksar El Kebir       | 2016.xls   ✅ 8,619 rows [xml]
[7/197] 00001A3F • ksar El Kebir       | 2017.xls   ✅ 8,734 rows [xml]
[8/197] 00001A3F • ksar El Kebir       | 2018.xls   ✅ 7,643 rows [xml]
[9/197] 00001A3F • ksar El Kebir       | 2022.xls   ✅ 8,646 rows [xml]
[10/197] 00001A3F • ksar El Kebir       | 2025.xls   ✅ 8,663 rows [xml]
[11/197] 00001A3F • ksar El Kebir       | 2012.xls   ✅ 5,482 rows [xml]
[12/197] 00001A3F • ksar El Kebir       | 2019.xls   ✅ 6,539 rows [xml]
[13/197] 00001A3F • ksar El Kebir       | 2024.xls   ✅ 8,648 rows [xml]

In [4]:
print(TERRAIN_ROOT)

/content/drive/MyDrive/MyDrive/Données Météo


In [5]:
import pandas as pd
import numpy as np

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 4 : Chargement Données Satellite (ERA5, Open-Meteo, NASA POWER)
# ═══════════════════════════════════════════════════════════════════════════
import gc
from typing import Tuple, Optional

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise les noms de colonnes pour faciliter la manipulation."""
    df.columns = [
        col.lower().replace(' ', '_').replace('.', '_').replace('-', '_')
        for col in df.columns
    ]
    df = df.rename(columns={
        't2m':                      'temperature',
        'total_precipitation_sum':  'precipitation',
        'tp':                       'precipitation',
        'date':                     'datetime',
    })
    return df


def convert_units(df: pd.DataFrame) -> pd.DataFrame:
    """Convertit les unités : Kelvin → Celsius pour les colonnes température."""
    temp_cols = [col for col in df.columns if 'temp' in col and df[col].max() > 100]
    for col in temp_cols:
        df[col] = df[col] - 273.15
    return df


def detect_geo_cols(df: pd.DataFrame, source_name: str = '') -> Tuple[Optional[str], Optional[str]]:
    """
    Détecte les colonnes lat/lon dans un DataFrame.
    Cherche des noms standard ('lat', 'latitude', 'lon', 'longitude')
    et des noms préfixés (ex: 'era5_lat', 'openmeteo_latitude') si source_name est fourni.
    """
    # Standard names
    possible_lat_names = ['lat', 'latitude']
    possible_lon_names = ['lon', 'longitude']

    # Add prefixed names based on common prefixes if source_name is available
    if source_name:
        possible_lat_names.extend([f'{source_name}_lat', f'{source_name}_latitude'])
        possible_lon_names.extend([f'{source_name}_lon', f'{source_name}_longitude'])
    # Also add known prefixed names from other sources to the search for generality
    possible_lat_names.extend(['nasa_lat', 'nasa_latitude', 'openmeteo_lat', 'openmeteo_latitude', 'era5_lat', 'era5_latitude'])
    possible_lon_names.extend(['nasa_lon', 'nasa_longitude', 'openmeteo_lon', 'openmeteo_longitude', 'era5_lon', 'era5_longitude'])

    # Remove duplicates and ensure order
    possible_lat_names = list(dict.fromkeys(possible_lat_names))
    possible_lon_names = list(dict.fromkeys(possible_lon_names))

    lat = next((c for c in df.columns if c in possible_lat_names), None)
    lon = next((c for c in df.columns if c in possible_lon_names), None)
    return lat, lon


def load_satellite_data(path: Path, source_name: str) -> pd.DataFrame:
    """Charge, standardise et préfixe les données satellite."""
    print(f"\n📡 Chargement {source_name}...")

    # Determine file type based on extension
    if path.suffix == '.csv':
        df = pd.read_csv(path)
    elif path.suffix == '.xlsx':
        df = pd.read_excel(path) # Use read_excel for .xlsx files
    else:
        raise ValueError(f"Unsupported file format for {source_name}: {path.suffix}")

    df = standardize_column_names(df)

    # NEW LOGIC: Identify and standardize geo columns BEFORE general prefixing
    actual_lat_col_in_df, actual_lon_col_in_df = detect_geo_cols(df, source_name='') # Generic search

    # Explicitly rename to 'latitude' and 'longitude' if found
    # and mark these as protected so they don't get 'source_name_' prefixed
    if actual_lat_col_in_df and actual_lon_col_in_df:
        if actual_lat_col_in_df != 'latitude': # Only rename if not already 'latitude'
            df = df.rename(columns={actual_lat_col_in_df: 'latitude'})
        if actual_lon_col_in_df != 'longitude': # Only rename if not already 'longitude'
            df = df.rename(columns={actual_lon_col_in_df: 'longitude'})
        print(f"  ✅ Geo-columns in {source_name} standardized to 'latitude', 'longitude'")
        final_protected_cols = {'datetime', 'latitude', 'longitude', 'source', 'region_id'}
    else:
        print(f"  ⚠️  No standard geo-columns found for {source_name}. Some geo-ops might fail if coordinates are missing.")
        # Fallback to original protected cols if no geo cols were found to standardize
        final_protected_cols = {'datetime', 'latitude', 'longitude', 'lat', 'lon', 'source', 'region_id'} # Fallback

    # ── Parse datetime ──────────────────────────────────────────────────────
    if 'datetime' not in df.columns:
        if all(c in df.columns for c in ['year', 'month', 'day']):
            df['datetime'] = pd.to_datetime(df[['year', 'month', 'day']])
        else:
            raise ValueError(f"[{source_name}] Impossible de construire la colonne 'datetime'. "
                             f"Colonnes disponibles : {list(df.columns)}")
    else:
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

    # Supprimer les lignes sans date valide
    n_before = len(df)
    df = df.dropna(subset=['datetime'])
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"  ⚠️  {n_dropped} lignes supprimées (datetime invalide)")

    # ── Conversions unités ──────────────────────────────────────────────────
    df = convert_units(df)

    # ── Marqueur source ─────────────────────────────────────────────────────
    df['source'] = source_name

    # ── Préfixage des colonnes (sauf colonnes protégées) ───────────────────
    rename_dict = {
        col: f"{source_name}_{col}"
        for col in df.columns
        if col not in final_protected_cols # Use the dynamically determined protected_cols
    }
    df = df.rename(columns=rename_dict)

    print(f"✅ {source_name} : {df.shape} | "
          f"Période : {df['datetime'].min()} → {df['datetime'].max()}")

    return df


# ── Chargement des 3 sources ────────────────────────────────────────────────
df_era5      = load_satellite_data(ERA5_PATH,      'era5')
df_openmeteo = load_satellite_data(OPENMETEO_PATH, 'openmeteo')
df_nasa      = load_satellite_data(NASA_PATH,      'nasa')

# Now, all dataframes should have 'latitude' and 'longitude' if they had geo data
# and these names will be consistent. So, we can set lat_col and lon_col directly.
lat_col = 'latitude'
lon_col = 'longitude'

# The previous block to detect lat_col, lon_col and raise ValueError is no longer needed
# as the above logic ensures consistent naming. If geo-cols were never found,
# assign_region_to_satellite will get 'latitude'/'longitude' but those columns won't exist in the DF,
# and it will raise an appropriate error. This is a more robust flow.

print(f"\n🌍 Colonnes géo utilisées : {lat_col}, {lon_col}")

# ── Sauvegarde ──────────────────────────────────────────────────────────────
df_era5.to_csv(PROCESSED / 'era5_processed.csv',         index=False)
df_openmeteo.to_csv(PROCESSED / 'openmeteo_processed.csv', index=False)
df_nasa.to_csv(PROCESSED / 'nasa_processed.csv',          index=False)

print("\n✅ Données satellite chargées et sauvegardées")
gc.collect()


📡 Chargement era5...
  ⚠️  No standard geo-columns found for era5. Some geo-ops might fail if coordinates are missing.
✅ era5 : (64995, 11) | Période : 2015-01-01 00:00:00 → 2025-12-30 00:00:00

📡 Chargement openmeteo...
  ✅ Geo-columns in openmeteo standardized to 'latitude', 'longitude'
✅ openmeteo : (1455360, 12) | Période : 2012-01-01 00:00:00 → 2024-12-31 23:00:00

📡 Chargement nasa...
  ✅ Geo-columns in nasa standardized to 'latitude', 'longitude'
✅ nasa : (2184, 19) | Période : 2012-01-01 00:00:00 → 2024-12-01 00:00:00

🌍 Colonnes géo utilisées : latitude, longitude

✅ Données satellite chargées et sauvegardées


110

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 5 : Fusion Multi-Sources avec 3 Approches
#             Version corrigée + optimisée mémoire
# ═══════════════════════════════════════════════════════════════════════════
import gc
import numpy as np
from scipy.spatial.distance import cdist

# ── Configuration ───────────────────────────────────────────────────────────
METADATA_EXCLUSIONS = [
    'source_file', 'annee', 'col_0', 'col_6', 'col_7', 'col_8',
    'col_9', 'col_10', 'col_11', 'lat', 'lon', 'latitude', 'longitude'
]


# ═══════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def haversine_distances(sat_coords: np.ndarray,
                        terrain_coords: np.ndarray) -> np.ndarray:
    """
    Calcul distances haversine (km) entre points satellite et terrain.
    Plus précis que l'approximation euclidienne × 111.
    """
    R = 6371.0
    lat1 = np.radians(sat_coords[:, 0][:, None])
    lon1 = np.radians(sat_coords[:, 1][:, None])
    lat2 = np.radians(terrain_coords[:, 0][None, :])
    lon2 = np.radians(terrain_coords[:, 1][None, :])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def ensure_terrain_coords(df_terrain: pd.DataFrame,
                           lat_col: str, lon_col: str) -> pd.DataFrame:
    """
    FIX : Ajoute des coordonnées fictives au terrain si absentes.
    En production : remplacer par un vrai fichier de métadonnées stations.
    """
    if lat_col in df_terrain.columns and lon_col in df_terrain.columns:
        return df_terrain

    print("🔧 Coordonnées manquantes dans df_terrain — ajout de coords fictives (Maroc)...")
    unique_regions = df_terrain['region_id'].unique()
    n = len(unique_regions)

    rng = np.random.default_rng(42)
    dummy_lats = np.linspace(27.0, 36.0, n)
    dummy_lons = np.linspace(-13.0, -1.0, n)
    rng.shuffle(dummy_lats)
    rng.shuffle(dummy_lons)

    coords_df = pd.DataFrame({
        'region_id': unique_regions,
        lat_col:     dummy_lats,
        lon_col:     dummy_lons,
    })

    df_terrain = df_terrain.merge(coords_df, on='region_id', how='left')
    print(f"   ✅ {n} régions enrichies avec coordonnées fictives")
    return df_terrain


def assign_region_to_satellite(df_sat: pd.DataFrame,
                                df_terrain: pd.DataFrame,
                                lat_col: str, lon_col: str,
                                max_distance_km: float = 50) -> pd.DataFrame:
    """
    Assigne chaque point satellite à la région terrain la plus proche
    via distance haversine. Filtre les points > max_distance_km.
    """
    # FIX : vérifier que les colonnes géo existent dans df_sat
    for col in [lat_col, lon_col]:
        if col not in df_sat.columns:
            raise ValueError(
                f"Colonne '{col}' absente du DataFrame satellite. "
                f"Colonnes disponibles : {list(df_sat.columns)}"
            )

    # Coordonnées terrain moyennes par région
    terrain_coords = (
        df_terrain.groupby('region_id')[[lat_col, lon_col]]
        .mean()
        .reset_index()
    )

    # Coordonnées satellite uniques
    sat_unique = df_sat[[lat_col, lon_col]].drop_duplicates().copy()

    # FIX : distances haversine (au lieu d'euclidien × 111 imprécis)
    dist_matrix = haversine_distances(
        sat_unique[[lat_col, lon_col]].values,
        terrain_coords[[lat_col, lon_col]].values
    )

    closest_idx      = dist_matrix.argmin(axis=1)
    closest_dist_km  = dist_matrix.min(axis=1)

    sat_unique['region_id']   = terrain_coords.iloc[closest_idx]['region_id'].values
    sat_unique['distance_km'] = closest_dist_km

    # Filtrage distance maximale
    sat_unique = sat_unique[sat_unique['distance_km'] <= max_distance_km]

    # FIX : drop propre de region_id existant avant merge pour éviter doublons
    if 'region_id' in df_sat.columns:
        df_sat = df_sat.drop(columns=['region_id'])

    df_sat = df_sat.merge(
        sat_unique[[lat_col, lon_col, 'region_id']],
        on=[lat_col, lon_col],
        how='left'
    )

    n_assigned = df_sat['region_id'].notna().sum()
    n_total    = len(df_sat)
    print(f"   ✅ {n_assigned}/{n_total} points assignés "
          f"({100 * n_assigned / n_total:.1f}%)")

    return df_sat


def prepare_terrain(df_terrain: pd.DataFrame) -> pd.DataFrame:
    """Prépare le DataFrame terrain : sélection colonnes, typage datetime."""
    cols_to_keep = ['datetime', 'region_id'] + [
        c for c in df_terrain.columns
        if c not in ['datetime', 'region_id'] + METADATA_EXCLUSIONS
    ]
    # FIX : éviter les KeyError si certaines colonnes listées n'existent pas
    cols_to_keep = [c for c in cols_to_keep if c in df_terrain.columns]

    df_t = df_terrain[cols_to_keep].copy()
    df_t['datetime'] = pd.to_datetime(df_t['datetime'], errors='coerce')
    return df_t


def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    """Réduit les types numériques pour économiser la RAM."""
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


# ═══════════════════════════════════════════════════════════════════════════
# ASSIGNATION RÉGIONS
# ═══════════════════════════════════════════════════════════════════════════

# FIX : s'assurer que df_terrain a des coordonnées avant l'assignation
df_terrain = ensure_terrain_coords(df_terrain, lat_col, lon_col)

# Add latitude and longitude to df_era5 by merging with df_terrain's coordinates
# This is necessary because ERA5 data in this context does not have inherent geo-coords
terrain_region_coords = df_terrain.groupby('region_id')[[lat_col, lon_col]].mean().reset_index()
if lat_col not in df_era5.columns:
    df_era5 = df_era5.merge(terrain_region_coords, on='region_id', how='left')
    print(f"  ✅ Coordinates from terrain assigned to df_era5 based on region_id.")

print("\n📍 Assignation des régions aux données satellite...")
print("  ERA5 :")
df_era5      = assign_region_to_satellite(df_era5,      df_terrain, lat_col, lon_col)
print("  Open-Meteo :")
df_openmeteo = assign_region_to_satellite(df_openmeteo, df_terrain, lat_col, lon_col)
print("  NASA :")
df_nasa      = assign_region_to_satellite(df_nasa,      df_terrain, lat_col, lon_col)


# ═══════════════════════════════════════════════════════════════════════════
# FUSION 3 APPROCHES
# ═══════════════════════════════════════════════════════════════════════════

def create_fusion_approaches(df_terrain: pd.DataFrame,
                              df_era5: pd.DataFrame,
                              df_openmeteo: pd.DataFrame,
                              df_nasa: pd.DataFrame) -> dict:
    """
    Crée 3 approches de fusion :
      A) Terrain + ERA5 uniquement
      B) Terrain + moyenne consensus (ERA5, Open-Meteo, NASA)
      C) Terrain + toutes sources séparément
    """
    MERGE_KEYS = ['datetime', 'region_id']

    # Typage datetime uniforme
    for df in [df_era5, df_openmeteo, df_nasa]:
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

    df_t = prepare_terrain(df_terrain)

    # ── Approche A : Terrain + ERA5 ─────────────────────────────────────────
    era5_cols = ['datetime', 'region_id'] + [
        c for c in df_era5.columns if c.startswith('era5_')
    ]
    # FIX : vérifier colonnes existantes
    era5_cols = [c for c in era5_cols if c in df_era5.columns]

    df_A = df_t.merge(
        df_era5[era5_cols],
        on=MERGE_KEYS, how='left'
    )
    df_A = downcast_df(df_A)
    print(f"\n✅ Approche A (Terrain + ERA5)       : {df_A.shape}")

    # ── Approche B : Terrain + Consensus ────────────────────────────────────
    # FIX : merge outer propre avec suffixes explicites
    df_sat = df_era5.merge(
        df_openmeteo, on=MERGE_KEYS, how='outer', suffixes=('', '_om')
    )
    df_sat = df_sat.merge(
        df_nasa, on=MERGE_KEYS, how='outer', suffixes=('', '_nasa')
    )

    # Calcul consensus pour variables communes
    common_vars = [
        'temperature', 'temp', 't2m',
        'precipitation', 'precip',
        'humidity', 'wind_speed', 'radiation'
    ]
    for var in common_vars:
        cols_var = [c for c in df_sat.columns if var in c.lower()]
        if len(cols_var) >= 2:
            df_sat[f'consensus_{var}'] = df_sat[cols_var].mean(axis=1, skipna=True)
            df_sat[f'std_{var}']       = df_sat[cols_var].std(axis=1,  skipna=True)

    df_B = df_t.merge(df_sat, on=MERGE_KEYS, how='left')
    df_B = downcast_df(df_B)
    print(f"✅ Approche B (Terrain + Consensus)  : {df_B.shape}")

    # Libération mémoire intermédiaire
    del df_sat
    gc.collect()

    # ── Approche C : Terrain + Toutes sources séparément ───────────────────
    df_C = df_t.merge(df_era5,      on=MERGE_KEYS, how='left')
    df_C = df_C.merge(df_openmeteo, on=MERGE_KEYS, how='left', suffixes=('', '_om'))
    df_C = df_C.merge(df_nasa,      on=MERGE_KEYS, how='left', suffixes=('', '_nasa'))
    df_C = downcast_df(df_C)
    print(f"✅ Approche C (Terrain + Toutes)     : {df_C.shape}")

    return {
        'A_terrain_era5':       df_A,
        'B_terrain_consensus':  df_B,
        'C_terrain_all':        df_C,
    }


# ── Exécution ───────────────────────────────────────────────────────────────
print("\n🔀 Création des 3 approches de fusion...")
fusion_datasets = create_fusion_approaches(
    df_terrain, df_era5, df_openmeteo, df_nasa
)

# ── Sauvegarde séquentielle avec libération mémoire ─────────────────────────
print("\n💾 Sauvegarde...")
for name, df in fusion_datasets.items():
    path = PROCESSED / f'fusion_{name}.csv'
    df.to_csv(path, index=False)
    print(f"   💾 {name} : {df.shape} → {path}")
    del df
    gc.collect()


gc.collect()

🔧 Coordonnées manquantes dans df_terrain — ajout de coords fictives (Maroc)...
   ✅ 24 régions enrichies avec coordonnées fictives
  ✅ Coordinates from terrain assigned to df_era5 based on region_id.

📍 Assignation des régions aux données satellite...
  ERA5 :
   ✅ 0/64995 points assignés (0.0%)
  Open-Meteo :
   ✅ 140280/1455360 points assignés (9.6%)
  NASA :
   ✅ 192/2184 points assignés (8.8%)

🔀 Création des 3 approches de fusion...

✅ Approche A (Terrain + ERA5)       : (1393724, 98)
✅ Approche B (Terrain + Consensus)  : (1393724, 138)
✅ Approche C (Terrain + Toutes)     : (1393724, 128)

💾 Sauvegarde...
   💾 A_terrain_era5 : (1393724, 98) → /content/drive/MyDrive/MyDrive/data_processed_v3/fusion_A_terrain_era5.csv
   💾 B_terrain_consensus : (1393724, 138) → /content/drive/MyDrive/MyDrive/data_processed_v3/fusion_B_terrain_consensus.csv
   💾 C_terrain_all : (1393724, 128) → /content/drive/MyDrive/MyDrive/data_processed_v3/fusion_C_terrain_all.csv


0

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 6 : Feature Engineering Complet + Lags Temporels  [RAM-safe]
# ═══════════════════════════════════════════════════════════════════════════

def create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée features temporelles cycliques — opère IN-PLACE pour éviter les copies
    """
    # Extraction base
    dt = df['datetime']
    df['year']       = dt.dt.year.astype('int16')
    df['month']      = dt.dt.month.astype('int8')
    df['day']        = dt.dt.day.astype('int8')
    df['hour']       = dt.dt.hour.astype('int8')
    df['dayofyear']  = dt.dt.dayofyear.astype('int16')
    df['weekofyear'] = dt.dt.isocalendar().week.astype('int8')
    df['dayofweek']  = dt.dt.dayofweek.astype('int8')

    # Encodage cyclique
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365).astype('float32')
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365).astype('float32')

    # Saisons → entiers int8 directement (évite get_dummies + float64)
    season_map = {12: 0, 1: 0, 2: 0,   # DJF
                   3: 1, 4: 1, 5: 1,   # MAM
                   6: 2, 7: 2, 8: 2,   # JJA
                   9: 3, 10: 3, 11: 3} # SON
    season_codes = df['month'].map(season_map).astype('int8')

    df['season_DJF'] = (season_codes == 0).astype('int8')
    df['season_MAM'] = (season_codes == 1).astype('int8')
    df['season_JJA'] = (season_codes == 2).astype('int8')
    df['season_SON'] = (season_codes == 3).astype('int8')

    # Interactions saisonnières
    df['hour_x_summer'] = (df['hour'] * df['season_JJA']).astype('float32')
    df['hour_x_winter'] = (df['hour'] * df['season_DJF']).astype('float32')

    print(f"   ✅ Features temporelles : {df.shape[1]} colonnes")
    return df


def create_meteorological_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée features météorologiques avancées — in-place, pas de .copy()
    """
    temp_cols = [c for c in df.columns
                 if any(x in c.lower() for x in ['temp', 't2m', 'lst'])]
    rh_cols   = [c for c in df.columns
                 if any(x in c.lower() for x in ['humidity', 'rh', 'humidite'])]

    # Conversion numérique si nécessaire
    for col in temp_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

    # VPD (uniquement si absent)
    if not any('vpd' in c.lower() for c in df.columns) and temp_cols and rh_cols:
        t  = df[temp_cols[0]].values
        rh = df[rh_cols[0]].values
        es = 0.611 * np.exp((17.27 * t) / (t + 237.3))
        df['vpd_calculated'] = (es - es * (rh / 100)).astype('float32')
        print(f"   ✅ VPD calculé depuis {temp_cols[0]} et {rh_cols[0]}")
        del es, t, rh  # libère mémoire intermédiaire

    # Amplitude thermique (si plusieurs sources temp)
    if len(temp_cols) >= 2:
        df['temp_amplitude']    = (df[temp_cols].max(axis=1)
                                   - df[temp_cols].min(axis=1)).astype('float32')
        df['temp_mean_sources'] = df[temp_cols].mean(axis=1).astype('float32')

    # Anomalie mensuelle
    if temp_cols and 'month' in df.columns:
        col = temp_cols[0]
        df[f'{col}_anomaly'] = (
            df[col] - df.groupby('month')[col].transform('mean')
        ).astype('float32')

    print(f"   ✅ Features météo : {df.shape[1]} colonnes")
    return df


def create_lag_features(df: pd.DataFrame, group_col: str = 'region_id') -> pd.DataFrame:
    """
    Lags temporels PAR RÉGION — in-place, types float32 dès la création
    """
    df.sort_values([group_col, 'datetime'], inplace=True)

    temp_cols = [c for c in df.columns
                 if any(x in c.lower() for x in ['temp', 't2m'])
                 and not any(k in c for k in ['anomaly', 'amplitude', 'lag', 'rolling'])]
    precip_cols = [c for c in df.columns
                   if any(x in c.lower() for x in ['precip', 'rain', 'pluie'])
                   and 'lag' not in c]

    for col in temp_cols[:2]:
        for lag in [1, 3, 6, 24]:
            df[f'{col}_lag_{lag}h'] = (
                df.groupby(group_col)[col].shift(lag).astype('float32')
            )

    for col in precip_cols[:1]:
        for lag in [1, 6, 24]:
            df[f'{col}_lag_{lag}h'] = (
                df.groupby(group_col)[col].shift(lag).astype('float32')
            )

    if temp_cols:
        df[f'{temp_cols[0]}_rolling_mean_24h'] = (
            df.groupby(group_col)[temp_cols[0]]
              .transform(lambda x: x.rolling(24, min_periods=1).mean())
              .astype('float32')
        )

    print(f"   ✅ Lags temporels : {df.shape[1]} colonnes")
    return df


def fill_missing_values_smart(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remplit les NaN — in-place, sans créer de colonnes intermédiaires
    """
    lag_cols     = [c for c in df.columns if 'lag' in c]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.difference(lag_cols)

    if lag_cols:
        df[lag_cols] = df[lag_cols].fillna(0)

    for col in numeric_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].mean(), inplace=True)

    print(f"   ✅ NaN remplis")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# Application — UN DATASET À LA FOIS pour éviter l'accumulation en RAM
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("🔧 FEATURE ENGINEERING  [mode RAM-safe : traitement séquentiel]")
print("=" * 80)

engineered_datasets = {}

for name, df in fusion_datasets.items():
    print(f"\n📊 {name}  ({df.shape[0]:,} lignes, {df.shape[1]} colonnes brutes)")

    # Toutes les étapes in-place sur le même objet
    df = create_temporal_features(df)
    df = create_meteorological_features(df)

    if 'region_id' in df.columns:
        df = create_lag_features(df, group_col='region_id')

    df = fill_missing_values_smart(df)

    # Float32 pour toutes les colonnes numériques (économie ~50 % vs float64)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype('float32')

    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"   ✅ {name} : {df.shape} | RAM : {mem_mb:.0f} MB")

    # Sauvegarde immédiate en parquet (4× plus léger que CSV)
    path = PROCESSED / f'engineered_{name}.parquet'
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    print(f"   💾 Sauvegardé → {path.name}")

    engineered_datasets[name] = df

    # Libération explicite — important si RAM < 16 GB
    gc.collect()

print("\n✅ Feature engineering terminé pour les 3 approches")
gc.collect()

🔧 FEATURE ENGINEERING  [mode RAM-safe : traitement séquentiel]

📊 A_terrain_era5  (1,393,724 lignes, 98 colonnes brutes)
   ✅ Features temporelles : 117 colonnes
   ✅ Features météo : 120 colonnes
   ✅ Lags temporels : 132 colonnes
   ✅ NaN remplis
   ✅ A_terrain_era5 : (1393724, 132) | RAM : 1224 MB
   💾 Sauvegardé → engineered_A_terrain_era5.parquet

📊 B_terrain_consensus  (1,393,724 lignes, 138 colonnes brutes)
   ✅ Features temporelles : 157 colonnes
   ✅ Features météo : 160 colonnes
   ✅ Lags temporels : 172 colonnes
   ✅ NaN remplis
   ✅ B_terrain_consensus : (1393724, 172) | RAM : 1466 MB
   💾 Sauvegardé → engineered_B_terrain_consensus.parquet

📊 C_terrain_all  (1,393,724 lignes, 128 colonnes brutes)
   ✅ Features temporelles : 147 colonnes
   ✅ Features météo : 150 colonnes
   ✅ Lags temporels : 162 colonnes
   ✅ NaN remplis
   ✅ C_terrain_all : (1393724, 162) | RAM : 1412 MB
   💾 Sauvegardé → engineered_C_terrain_all.parquet

✅ Feature engineering terminé pour les 3 approche

0

In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 7 OPTIMISÉE : Leakage Detection LIGHT (No RAM crash)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd # Added this line
from typing import List, Tuple

def detect_and_remove_leakage_light(df: pd.DataFrame,
                                   target_col: str,
                                   correlation_threshold: float = 0.97,
                                   max_features: int = 40,
                                   sample_frac: float = 0.2) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    """
    Version optimisée RAM :
    - Pas de matrice NxN
    - Corrélation feature vs target seulement
    - Sampling data
    - Limitation features
    """

    print(f"\n⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)")

    # Sampling pour réduire RAM
    if len(df) > 50000:
        df = df.sample(frac=sample_frac, random_state=42)
        print(f"   🔻 Sampling appliqué : {len(df)} lignes")

    # Colonnes numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if target_col not in numeric_cols:
        print(f"⚠️ Target non valide")
        return df, [], pd.DataFrame()

    # Limiter features
    numeric_cols = numeric_cols[:max_features]

    # Calcul corrélation simple (feature vs target uniquement)
    correlations = {}

    target_values = df[target_col]

    for col in numeric_cols:
        if col == target_col:
            continue
        try:
            correlations[col] = abs(df[col].corr(target_values))
        except:
            correlations[col] = 0

    correlations = pd.Series(correlations).sort_values(ascending=False)

    # Détection leakage
    leakage_features = correlations[correlations > correlation_threshold].index.tolist()

    # Rapport
    correlation_report = pd.DataFrame({
        'feature': correlations.index,
        'correlation_with_target': correlations.values,
        'is_leakage': correlations.index.isin(leakage_features)
    })

    # Suppression
    df_clean = df.drop(columns=leakage_features, errors='ignore')

    print(f"   ❌ Leakage détecté : {len(leakage_features)} features")
    print(f"   ✅ Shape : {df.shape} → {df_clean.shape}")

    return df_clean, leakage_features, correlation_report


def select_features_light(df: pd.DataFrame,
                         target_col: str,
                         max_features: int = 30) -> Tuple[pd.DataFrame, List[str]]:
    """
    Sélection ultra light pour éviter crash
    """

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude = [target_col, 'datetime', 'region_id', 'year']
    feature_cols = [c for c in numeric_cols if c not in exclude]

    # Supprimer NaN full
    feature_cols = [c for c in feature_cols if not df[c].isnull().all()]

    # 🔻 LIMITATION FORTE
    feature_cols = feature_cols[:max_features]

    print(f"⚡ Features limitées à {len(feature_cols)}")

    return df, feature_cols


# ═══════════════════════════════════════════════════════════════════════════
# APPLICATION LIGHT
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("⚡ VERSION OPTIMISÉE ANTI-RAM")
print("=" * 80)

cleaned_datasets = {}
leakage_reports = {}

TARGETS = {
    'temperature': [c for c in engineered_datasets['A_terrain_era5'].columns
                    if 'temperature' in c.lower() or 't2m' in c.lower()][0],
    'precipitation': [c for c in engineered_datasets['A_terrain_era5'].columns
                      if 'precip' in c.lower() or 'rain' in c.lower()][0]
}

for name, df in engineered_datasets.items():

    print(f"\n📊 Dataset : {name}")

    df_clean = df.copy()
    all_leakage = []

    for target_name, target_col in TARGETS.items():

        if target_col not in df.columns:
            continue

        print(f"   🎯 Target : {target_name}")

        df_clean, leakage, report = detect_and_remove_leakage_light(
            df_clean,
            target_col
        )

        all_leakage.extend(leakage)
        leakage_reports[f"{name}_{target_name}"] = report

    cleaned_datasets[name] = df_clean

    print(f"   ✅ Final shape : {df.shape} → {df_clean.shape}")

gc.collect()

⚡ VERSION OPTIMISÉE ANTI-RAM

📊 Dataset : A_terrain_era5
   🎯 Target : temperature

⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)
   🔻 Sampling appliqué : 278745 lignes
   ❌ Leakage détecté : 3 features
   ✅ Shape : (278745, 132) → (278745, 129)
   🎯 Target : precipitation

⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)
   🔻 Sampling appliqué : 55749 lignes
⚠️ Target non valide
   ✅ Final shape : (1393724, 132) → (55749, 129)

📊 Dataset : B_terrain_consensus
   🎯 Target : temperature

⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)
   🔻 Sampling appliqué : 278745 lignes
   ❌ Leakage détecté : 3 features
   ✅ Shape : (278745, 172) → (278745, 169)
   🎯 Target : precipitation

⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)
   🔻 Sampling appliqué : 55749 lignes
⚠️ Target non valide
   ✅ Final shape : (1393724, 172) → (55749, 169)

📊 Dataset : C_terrain_all
   🎯 Target : temperature

⚡ MODE LIGHT ACTIVÉ (anti-crash RAM)
   🔻 Sampling appliqué : 278745 lignes
   ❌ Leakage détecté : 3 features
   ✅ Shape : (278745, 162) → (278745, 

0

In [11]:
import gc
import psutil
import numpy as np
import matplotlib
matplotlib.use('Agg')  # FIX : pas de rendu interactif → économise RAM
import matplotlib.pyplot as plt
from typing import Tuple, List
import pandas as pd # Ensure pandas is imported

def log_ram(label: str):
    used = psutil.Process().memory_info().rss / 1024**2
    print(f"  🧠 RAM [{label}] : {used:.0f} MB")


def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    """Réduit les types numériques pour économiser la RAM."""
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


def temporal_split(df: pd.DataFrame,
                   train_ratio: float = 0.6,
                   val_ratio:   float = 0.2,
                   test_ratio:  float = 0.2
                   ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split temporel strict (pas de shuffle).
    ⚠️ CRITIQUE pour séries temporelles :
       Train = passé | Val = futur proche | Test = futur lointain
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.01, \
        "Ratios doivent sommer à 1"

    df = df.sort_values('datetime').reset_index(drop=True)

    n         = len(df)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    # FIX : on coupe par index sans .copy() immédiat → on copie après sélection features
    train = df.iloc[:train_end]
    val   = df.iloc[train_end:val_end]
    test  = df.iloc[val_end:]

    print(f"✅ Split temporel :")
    print(f"   Train : {len(train):>7,} ({100*train_ratio:.0f}%) | "
          f"{train['datetime'].min()} → {train['datetime'].max()}")
    print(f"   Val   : {len(val):>7,} ({100*val_ratio:.0f}%) | "
          f"{val['datetime'].min()} → {val['datetime'].max()}")
    print(f"   Test  : {len(test):>7,} ({100*test_ratio:.0f}%) | "
          f"{test['datetime'].min()} → {test['datetime'].max()}")

    return train, val, test


def prepare_features_target(df: pd.DataFrame,
                             target_col: str,
                             feature_cols: List[str],
                             drop_na_target: bool = True
                             ) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Sépare features et target.
    FIX : sélection colonnes AVANT .copy() pour éviter copie inutile du DF complet.
    """
    if target_col not in df.columns:
        raise ValueError(f"Target '{target_col}' absente. "
                         f"Colonnes dispo : {list(df.columns)[:10]}...")

    # FIX : sélectionner uniquement les colonnes nécessaires avant tout traitement
    available_features = [c for c in feature_cols if c in df.columns]
    missing_features   = set(feature_cols) - set(available_features)
    if missing_features:
        print(f"  ⚠️  {len(missing_features)} features absentes : "
              f"{list(missing_features)[:5]}...")

    cols_needed = available_features + [target_col]
    df_small = df[cols_needed].copy()  # FIX : copie partielle, pas tout le DF

    # Typage numérique de la target
    df_small[target_col] = pd.to_numeric(df_small[target_col], errors='coerce')

    # Filtrage NaN target
    if drop_na_target:
        n_before  = len(df_small)
        df_small  = df_small[df_small[target_col].notna()]
        n_dropped = n_before - len(df_small)
        if n_dropped:
            print(f"  🔧 {n_dropped} lignes supprimées (target NaN)")

    X = df_small[available_features]
    y = df_small[target_col]

    print(f"  ✅ X: {X.shape} | y: {y.shape} | NaN dans X: {X.isna().sum().sum()}")

    return X, y


def plot_distributions(y_sets: dict, title: str, log_scale: bool = False,
                       save_path=None):
    """
    FIX : Génère et sauvegarde le plot PUIS ferme la figure immédiatement
    pour libérer la mémoire matplotlib.
    """
    fig, ax = plt.subplots(figsize=(10, 4))

    arrays = []
    labels = []
    for label, y in y_sets.items():
        arr = y.values
        if log_scale:
            # Ensure values are non-negative before log1p
            arr = np.clip(arr, 0, None)
            arr = np.log1p(arr)
        arrays.append(arr)
        labels.append(label)

    ax.hist(arrays, bins=50, label=labels, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('log(1 + valeur)' if log_scale else 'valeur')
    ax.legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')  # FIX : dpi 100 < 150
        print(f"  💾 Figure sauvegardée : {save_path}")

    plt.close(fig)   # FIX : fermeture explicite → libère mémoire matplotlib
    del fig, ax, arrays
    gc.collect()


def select_features_for_training(df: pd.DataFrame,
                         target_col: str,
                         max_features: int = 30) -> Tuple[pd.DataFrame, List[str]]:
    """
    Sélection light de features pour l'entraînement.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude = [target_col, 'datetime', 'region_id', 'year']
    feature_cols = [c for c in numeric_cols if c not in exclude]

    # Supprimer NaN full
    feature_cols = [c for c in feature_cols if not df[c].isnull().all()]

    # LIMITATION FORTE
    # This part should ideally be adjusted based on actual feature importance
    # but for now, we limit to avoid errors with a potentially large number of features
    feature_cols = feature_cols[:max_features]

    print(f"   ⚡ Features limitées à {len(feature_cols)} pour l'entraînement")

    return df, feature_cols

# ═══════════════════════════════════════════════════════════════════════════
# APPLICATION
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("📊 SPLIT DONNÉES")
print("=" * 80)
log_ram("début cellule 8")

# ── Chargement minimal du dataset principal ─────────────────────────────────
# FIX : charger depuis CSV sauvegardé + downcast immédiat
# au lieu de garder cleaned_datasets en RAM
if 'cleaned_datasets' in dir() and 'A_terrain_era5' in cleaned_datasets:
    df_main = cleaned_datasets['A_terrain_era5'].copy() # Add .copy() here to avoid SettingWithCopyWarning
    print("✅ Dataset chargé depuis cleaned_datasets en mémoire")
else:
    print("📂 Chargement depuis CSV sauvegardé...")
    df_main = pd.read_csv(PROCESSED / 'fusion_A_terrain_era5.csv',
                          parse_dates=['datetime'])

df_main = downcast_df(df_main)
log_ram("après chargement df_main")

# ── Identification features ─────────────────────────────────────────────────
_, feature_cols_temp   = select_features_for_training(df_main, TARGETS['temperature'])
_, feature_cols_precip = select_features_for_training(df_main, TARGETS['precipitation'])

print(f"\n🎯 Features température   : {len(feature_cols_temp)}")
print(f"🎯 Features précipitation : {len(feature_cols_precip)}")

# ── Split temporel ──────────────────────────────────────────────────────────
print()
train, val, test = temporal_split(df_main, 0.6, 0.2, 0.2)

# FIX : libérer df_main dès que les splits sont créés (vues, pas copies)
# On force la copie uniquement au moment de prepare_features_target
log_ram("après split")

# ── Température ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🌡️  TEMPÉRATURE")
print("="*60)

X_train_temp,  y_train_temp  = prepare_features_target(train, TARGETS['temperature'], feature_cols_temp)
X_val_temp,    y_val_temp    = prepare_features_target(val,   TARGETS['temperature'], feature_cols_temp)
X_test_temp,   y_test_temp   = prepare_features_target(test,  TARGETS['temperature'], feature_cols_temp)
log_ram("après features température")

# ── Précipitation ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🌧️  PRÉCIPITATION")
print("="*60)

X_train_precip, y_train_precip = prepare_features_target(train, TARGETS['precipitation'], feature_cols_precip)
X_val_precip,   y_val_precip   = prepare_features_target(val,   TARGETS['precipitation'], feature_cols_precip)
X_test_precip,  y_test_precip  = prepare_features_target(test,  TARGETS['precipitation'], feature_cols_precip)
log_ram("après features précipitation")

# FIX : libérer les splits bruts — on n'a plus besoin que de X/y
del train, val, test, df_main
gc.collect()
log_ram("après libération splits bruts")

# ── Sauvegarde structure splits ─────────────────────────────────────────────
splits = {
    'temperature': {
        'X_train': X_train_temp,   'y_train': y_train_temp,
        'X_val':   X_val_temp,     'y_val':   y_val_temp,
        'X_test':  X_test_temp,    'y_test':  y_test_temp,
        'feature_cols': feature_cols_temp,
    },
    'precipitation': {
        'X_train': X_train_precip,  'y_train': y_train_precip,
        'X_val':   X_val_precip,    'y_val':   y_val_precip,
        'X_test':  X_test_precip,   'y_test':  y_test_precip,
        'feature_cols': feature_cols_precip,
    },
}

# ── Visualisations (une par une, fermées immédiatement) ─────────────────────
print("\n📊 Génération des visualisations...")

plot_distributions(
    {'Train': y_train_temp, 'Val': y_val_temp, 'Test': y_test_temp},
    title      = 'Distribution Température (°C)',
    log_scale  = False,
    save_path  = PROCESSED / 'dist_temperature_splits.png'
)

plot_distributions(
    {'Train': y_train_precip, 'Val': y_val_precip, 'Test': y_test_precip},
    title      = 'Distribution Précipitation (log mm)',
    log_scale  = True,
    save_path  = PROCESSED / 'dist_precipitation_splits.png'
)

log_ram("fin cellule 8")
print("\n✅ Splits créés et sauvegardés")
gc.collect()


📊 SPLIT DONNÉES
  🧠 RAM [début cellule 8] : 4431 MB
✅ Dataset chargé depuis cleaned_datasets en mémoire
  🧠 RAM [après chargement df_main] : 4431 MB
   ⚡ Features limitées à 30 pour l'entraînement
   ⚡ Features limitées à 30 pour l'entraînement

🎯 Features température   : 30
🎯 Features précipitation : 30

✅ Split temporel :
   Train :  33,449 (60%) | 2012-01-06 00:00:00 → 2022-08-16 07:00:00
   Val   :  11,150 (20%) | 2022-08-16 09:00:00 → 2024-05-25 02:00:00
   Test  :  11,150 (20%) | 2024-05-25 06:00:00 → 2025-12-31 18:00:00
  🧠 RAM [après split] : 4395 MB

🌡️  TEMPÉRATURE
  🔧 21101 lignes supprimées (target NaN)
  ✅ X: (12348, 30) | y: (12348,) | NaN dans X: 123404
  🔧 8996 lignes supprimées (target NaN)
  ✅ X: (2154, 30) | y: (2154,) | NaN dans X: 21540
  🔧 8798 lignes supprimées (target NaN)
  ✅ X: (2352, 30) | y: (2352,) | NaN dans X: 23520
  🧠 RAM [après features température] : 4395 MB

🌧️  PRÉCIPITATION
  🔧 12650 lignes supprimées (target NaN)
  ✅ X: (20799, 30) | y: (20799,) |

0

In [13]:
# ✅ Simplified training (XGBoost only + reduced data)

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd # Ensure pandas is imported

# The 'df' variable in this cell refers to the global 'df' which was last assigned
# in the loop in Cell 6/7. It holds the 'df_clean' from the last iteration of leakage detection,
# which is already a sampled subset and may contain NaNs.
# For consistency and clarity, let's explicitly load the 'A_terrain_era5' dataset from engineered_datasets,
# as done in Cell 8, and then apply the sampling here. This ensures we start with
# the intended processed data before any further sampling or leakage detection.

# Assuming 'engineered_datasets' is available globally (it is in the notebook state)
# We take one of the engineered datasets that underwent feature engineering and NaN filling.
if 'engineered_datasets' in globals() and 'A_terrain_era5' in engineered_datasets:
    base_df = engineered_datasets['A_terrain_era5'].copy()
else:
    # Fallback: if engineered_datasets is not present, load from saved parquet
    print("Warning: 'engineered_datasets' not found. Loading 'engineered_A_terrain_era5.parquet' from disk.")
    base_df = pd.read_parquet(PROCESSED / 'engineered_A_terrain_era5.parquet')


# 🔻 Reduce dataset size to avoid RAM crash
# Sample from the base_df, not the 'df' that might be an already sampled 'df_clean'
df_small = base_df.sample(frac=0.3, random_state=42)

# Define target column explicitly. Using the temperature target from the overall notebook context.
target_col = 'hc_air_temperature_c'

if target_col not in df_small.columns:
    raise ValueError(f"Target column '{target_col}' not found in the sampled DataFrame.")

# 🔻 Reduce features (keep only first 30 numeric columns relevant for modeling)
# Exclude target, datetime, region_id, and year from features
numeric_cols = df_small.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    col for col in numeric_cols
    if col not in [target_col, 'year', 'datetime', 'region_id'] # datetime/region_id might be non-numeric but good to exclude
]
# Take the first 30 features from the filtered list
feature_cols = feature_cols[:30]

if not feature_cols:
    raise ValueError("No valid numeric features found after selection. Cannot train model.")

X = df_small[feature_cols].copy() # Features
y = df_small[target_col].copy()   # Target

# --- IMPORTANT: Handle NaNs in X and y ---
# 1. Filter out rows where the target (y) is NaN
initial_len = len(X)
valid_indices = y.notna()
X = X[valid_indices]
y = y[valid_indices]

if X.empty:
    raise ValueError("No valid data left after dropping NaN target values.")
print(f"Dropped {initial_len - len(X)} rows due to NaN target values.")

# 2. Fill any remaining NaNs in the feature set (X). Use 0 for tree models like XGBoost.
X.fillna(0, inplace=True)

# Ensure X and y are aligned after cleaning
if len(X) != len(y):
    raise ValueError("X and y lengths mismatch after NaN handling. This should not happen if valid_indices was applied correctly.")

# Split
# Check if X or y are empty after cleaning
if X.empty or y.empty:
    raise ValueError("X or y is empty after preprocessing. Cannot split data.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# 🚀 XGBoost (light config)
model = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.7,
    colsample_bytree=0.7,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"✅ XGBoost RMSE: {rmse:.3f}")


Dropped 290820 rows due to NaN target values.
X_train shape: (101837, 30), y_train shape: (101837,)
X_test shape: (25460, 30), y_test shape: (25460,)
✅ XGBoost RMSE: 3.560


In [24]:
# ──────────────────────────────────────────────────────────
# CELLULE 9B : Entra&#238;nement XGBoost — PR&#201;CIPITATION (sans Random Forest)
# ──────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import joblib
import gc
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path

# ──────────────────────────────────────────────────────────
# Nettoyage des donn&#233;es pr&#233;cipitation
# ──────────────────────────────────────────────────────────

def clean_precip_split(X, y, split_name):
    y = pd.Series(y).reset_index(drop=True)
    X = pd.DataFrame(X).reset_index(drop=True)

    mask_nan = y.isna()
    mask_neg = y < 0
    mask_bad = mask_nan | mask_neg

    n_nan = mask_nan.sum()
    n_neg = mask_neg.sum()
    n_total = len(y)

    print(f"   [{split_name}] NaN : {n_nan} | N&#233;gatifs : {n_neg} "
          f"| Conserv&#233;s : {n_total - mask_bad.sum()}/{n_total} "
          f"({(1 - mask_bad.mean())*100:.1f}%)")

    return X[~mask_bad].values, y[~mask_bad].values


print("=" * 60)
print("  DIAGNOSTIC PR&#201;CIPITATION")
print("=" * 60)

X_train_p, y_train_p = clean_precip_split(
    splits['precipitation']['X_train'],
    splits['precipitation']['y_train'],
    'train'
)

X_val_p, y_val_p = clean_precip_split(
    splits['precipitation']['X_val'],
    splits['precipitation']['y_val'],
    'val'
)

# log1p (gestion des z&#233;ros)
y_train_precip_log = np.log1p(y_train_p)
y_val_precip_log   = np.log1p(y_val_p)

assert not np.isnan(y_train_precip_log).any()
assert not np.isnan(y_val_precip_log).any()

print("   ✅ Donn&#233;es propres — log1p appliqu&#233; sans NaN")


# ──────────────────────────────────────────────────────────
# XGBoost + Optuna
# ──────────────────────────────────────────────────────────

def train_xgboost(X_train, y_train, X_val, y_val, n_trials=50):

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    def objective(trial):

        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "booster": trial.suggest_categorical("booster", ["gbtree", "dart"]),
            "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "seed": 42,
        }

        params["max_depth"] = trial.suggest_int("max_depth", 3, 10)
        params["eta"] = trial.suggest_float("eta", 1e-3, 0.3, log=True)
        params["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)

        if params["booster"] == "dart":
            params["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
            params["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
            params["rate_drop"] = trial.suggest_float("rate_drop", 0.0, 0.5)
            params["skip_drop"] = trial.suggest_float("skip_drop", 0.0, 0.5)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=300,
            evals=[(dval, "val")],
            early_stopping_rounds=20,
            verbose_eval=False
        )

        return model.best_score

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    print("\n┢ Meilleurs hyperparam&#232;tres XGBoost:")
    print(best_params)

    best_model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=500,
        evals=[(dval, "val")],
        early_stopping_rounds=20,
        verbose_eval=False
    )

    # ───────── m&#233;triques
    y_train_pred = np.expm1(best_model.predict(dtrain))
    y_val_pred   = np.expm1(best_model.predict(dval))

    metrics = {
        "train_rmse": np.sqrt(mean_squared_error(y_train_p, y_train_pred)),
        "val_rmse":   np.sqrt(mean_squared_error(y_val_p, y_val_pred)),
        "train_r2":   r2_score(y_train_p, y_train_pred),
        "val_r2":     r2_score(y_val_p, y_val_pred),
        "best_params": best_params
    }

    return best_model, metrics


# ──────────────────────────────────────────────────────────
# TRAINING
# ──────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  PR&#201;CIPITATION — XGBoost (OPTUNA)")
print("=" * 60)

xgb_precip, metrics_xgb_precip = train_xgboost(
    X_train_p, y_train_precip_log,
    X_val_p, y_val_precip_log,
    n_trials=60
)

# ──────────────────────────────────────────────────────────
# &#201;valuation finale
# ──────────────────────────────────────────────────────────

dtrain = xgb.DMatrix(X_train_p)
dval   = xgb.DMatrix(X_val_p)

y_train_pred = np.expm1(xgb_precip.predict(dtrain))
y_val_pred   = np.expm1(xgb_precip.predict(dval))

print("\n✅ XGBoost Pr&#233;cipitation :")
print(f"   Train RMSE: {np.sqrt(mean_squared_error(y_train_p, y_train_pred)):.3f} mm | R&#178;: {r2_score(y_train_p, y_train_pred):.3f}")
print(f"   Val   RMSE: {np.sqrt(mean_squared_error(y_val_p, y_val_pred)):.3f} mm | R&#178;: {r2_score(y_val_p, y_val_pred):.3f}")


# ──────────────────────────────────────────────────────────
# Sauvegarde
# ──────────────────────────────────────────────────────────

PROCESSED = Path("processed")
PROCESSED.mkdir(exist_ok=True) # Create the directory if it doesn't exist

xgb_precip.save_model(str(PROCESSED / "model_xgb_precipitation.json"))

print("\n✅ Mod&#232;le XGBoost pr&#233;cipitation sauvegard&#233;")

gc.collect()

[I 2026-05-06 18:19:09,860] A new study created in memory with name: no-name-1d3aeca3-e6fc-4419-8b92-62bf460a32bc


  DIAGNOSTIC PR&#201;CIPITATION
   [train] NaN : 0 | N&#233;gatifs : 45 | Conserv&#233;s : 20754/20799 (99.8%)
   [val] NaN : 0 | N&#233;gatifs : 14 | Conserv&#233;s : 5326/5340 (99.7%)
   ✅ Donn&#233;es propres — log1p appliqu&#233; sans NaN

  PR&#201;CIPITATION — XGBoost (OPTUNA)


[I 2026-05-06 18:19:11,446] Trial 0 finished with value: 1.1952558587547009 and parameters: {'booster': 'gbtree', 'lambda': 0.3969138072018704, 'alpha': 1.8428838422831606e-05, 'subsample': 0.7738835580856751, 'colsample_bytree': 0.9699660733371662, 'min_child_weight': 2, 'max_depth': 5, 'eta': 0.005323430621364684, 'gamma': 2.317456152007744e-08}. Best is trial 0 with value: 1.1952558587547009.
[I 2026-05-06 18:19:16,447] Trial 1 finished with value: 2.1257118592708695 and parameters: {'booster': 'gbtree', 'lambda': 0.00037722364506933545, 'alpha': 0.04369103192691918, 'subsample': 0.9677624913638291, 'colsample_bytree': 0.5708164079184828, 'min_child_weight': 10, 'max_depth': 10, 'eta': 0.0010360284795402626, 'gamma': 1.920940780351889e-05}. Best is trial 0 with value: 1.1952558587547009.
[I 2026-05-06 18:20:08,051] Trial 2 finished with value: 1.1716590179546862 and parameters: {'booster': 'dart', 'lambda': 0.0005506186178098741, 'alpha': 5.53791581070775e-08, 'subsample': 0.5566756


┢ Meilleurs hyperparam&#232;tres XGBoost:
{'booster': 'dart', 'lambda': 4.975357044613956e-06, 'alpha': 0.0002256655086432233, 'subsample': 0.7633610914858238, 'colsample_bytree': 0.7564389868401178, 'min_child_weight': 10, 'max_depth': 8, 'eta': 0.05001140424700893, 'gamma': 0.000570501346138024, 'sample_type': 'weighted', 'normalize_type': 'forest', 'rate_drop': 0.02756201140468792, 'skip_drop': 0.49041852802903446}

✅ XGBoost Pr&#233;cipitation :
   Train RMSE: 172.354 mm | R&#178;: 0.649
   Val   RMSE: 171.246 mm | R&#178;: 0.691

✅ Mod&#232;le XGBoost pr&#233;cipitation sauvegard&#233;


6967

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE : ENTRAÎNEMENT TEMPÉRATURE — XGBOOST (FAST & NO RAM CRASH)
# ═══════════════════════════════════════════════════════════════════════════

import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import gc

print("=" * 60)
print("🌡️ TEMPÉRATURE — XGBOOST OPTIMISÉ")
print("=" * 60)

# ──────────────────────────────────────────────────────────────────────────
# 1. Récupération données
# ──────────────────────────────────────────────────────────────────────────

X_train = splits['temperature']['X_train']
y_train = splits['temperature']['y_train']

X_val = splits['temperature']['X_val']
y_val = splits['temperature']['y_val']

# ──────────────────────────────────────────────────────────────────────────
# 2. Nettoyage NaN
# ──────────────────────────────────────────────────────────────────────────

def clean_temp_split(X, y, name):
    y = pd.Series(y).reset_index(drop=True)
    X = pd.DataFrame(X).reset_index(drop=True)

    mask = y.notna()

    print(f"[{name}] Conservé : {mask.sum()}/{len(y)} ({mask.mean()*100:.1f}%)")

    return X[mask].values, y[mask].values

X_train, y_train = clean_temp_split(X_train, y_train, "train")
X_val, y_val     = clean_temp_split(X_val, y_val, "val")

# ──────────────────────────────────────────────────────────────────────────
# 3. RÉDUCTION DATA (ANTI RAM)
# ──────────────────────────────────────────────────────────────────────────

MAX_ROWS = 200000  # 🔻 ajuste si besoin

if len(X_train) > MAX_ROWS:
    idx = np.random.choice(len(X_train), MAX_ROWS, replace=False)
    X_train = X_train[idx]
    y_train = y_train[idx]
    print(f"🔻 Sampling train → {len(X_train)} lignes")

# ──────────────────────────────────────────────────────────────────────────
# 4. LIMITATION FEATURES
# ──────────────────────────────────────────────────────────────────────────

MAX_FEATURES = 30

if X_train.shape[1] > MAX_FEATURES:
    X_train = X_train[:, :MAX_FEATURES]
    X_val   = X_val[:, :MAX_FEATURES]
    print(f"🔻 Features limitées à {MAX_FEATURES}")

# ──────────────────────────────────────────────────────────────────────────
# 5. DMatrix (XGBoost optimisé mémoire)
# ──────────────────────────────────────────────────────────────────────────

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

# ──────────────────────────────────────────────────────────────────────────
# 6. PARAMÈTRES OPTIMISÉS (rapides + stables)
# ──────────────────────────────────────────────────────────────────────────

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',   # ⚡ très important (rapide + low RAM)
    'n_jobs': -1,
    'seed': 42
}

# ──────────────────────────────────────────────────────────────────────────
# 7. TRAINING
# ──────────────────────────────────────────────────────────────────────────

model = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=20,
    verbose_eval=50
)

# ──────────────────────────────────────────────────────────────────────────
# 8. ÉVALUATION
# ──────────────────────────────────────────────────────────────────────────

y_train_pred = model.predict(dtrain)
y_val_pred   = model.predict(dval)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse   = np.sqrt(mean_squared_error(y_val, y_val_pred))

train_r2 = r2_score(y_train, y_train_pred)
val_r2   = r2_score(y_val, y_val_pred)

print("\n✅ RÉSULTATS TEMPÉRATURE :")
print(f"   Train RMSE: {train_rmse:.3f} °C | R²: {train_r2:.3f}")
print(f"   Val   RMSE: {val_rmse:.3f} °C | R²: {val_r2:.3f}")

# ──────────────────────────────────────────────────────────────────────────
# 9. SAUVEGARDE
# ──────────────────────────────────────────────────────────────────────────

model.save_model(str(PROCESSED / 'model_xgb_temperature.json'))

metrics_temp = {
    'train_rmse': train_rmse,
    'val_rmse': val_rmse,
    'train_r2': train_r2,
    'val_r2': val_r2
}

joblib.dump(metrics_temp, PROCESSED / 'metrics_temperature.pkl')

print("\n💾 Modèle température sauvegardé")

gc.collect()

🌡️ TEMPÉRATURE — XGBOOST OPTIMISÉ
[train] Conservé : 12348/12348 (100.0%)
[val] Conservé : 2154/2154 (100.0%)
[0]	train-rmse:7.29271	val-rmse:6.97438
[50]	train-rmse:0.39476	val-rmse:0.55727
[100]	train-rmse:0.21974	val-rmse:0.37334
[150]	train-rmse:0.16328	val-rmse:0.31900
[199]	train-rmse:0.13754	val-rmse:0.29569

✅ RÉSULTATS TEMPÉRATURE :
   Train RMSE: 0.138 °C | R²: 1.000
   Val   RMSE: 0.296 °C | R²: 0.999

💾 Modèle température sauvegardé


71

In [25]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 10 : Comparaison Modèles (VERSION CORRIGÉE + SAFE)
# ═══════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("📊 COMPARAISON DES MODÈLES")
print("=" * 80)


# ──────────────────────────────────────────────────────────────────────────
# 🔒 Sécurisation des métriques (évite les 0 silencieux)
# ──────────────────────────────────────────────────────────────────────────

def safe_metrics(metrics, name):
    if metrics is None:
        print(f"⚠️ Warning: {name} est None")
        return {
            'train_rmse': np.nan,
            'val_rmse': np.nan,
            'train_r2': np.nan,
            'val_r2': np.nan
        }
    return metrics


# ──────────────────────────────────────────────────────────────────────────
# 🔥 STRUCTURE PROPRE (VERSION CORRIGÉE DEMANDÉE)
# ──────────────────────────────────────────────────────────────────────────

metrics_all = {
    "temperature": {
        "xgb": safe_metrics(metrics_xgb_temp, "metrics_xgb_temp")
    },
    "precipitation": {
        "xgb": safe_metrics(metrics_xgb_precip, "metrics_xgb_precip")
    }
}


# ──────────────────────────────────────────────────────────────────────────
# 📊 Tableau comparatif
# ──────────────────────────────────────────────────────────────────────────

comparison_data = []

for target in metrics_all:
    for model_name in metrics_all[target]:

        metrics = metrics_all[target][model_name]

        comparison_data.append({
            'Target': target.capitalize(),
            'Modèle': 'XGBoost',
            'Train RMSE': metrics['train_rmse'],
            'Val RMSE': metrics['val_rmse'],
            'Train R²': metrics['train_r2'],
            'Val R²': metrics['val_r2'],
            'Overfitting': metrics['train_rmse'] - metrics['val_rmse']
        })


df_comparison = pd.DataFrame(comparison_data)

print("\n📋 TABLEAU COMPARATIF :\n")
print(df_comparison.to_string(index=False))


# ──────────────────────────────────────────────────────────────────────────
# 💾 Sauvegarde tableau
# ──────────────────────────────────────────────────────────────────────────

df_comparison.to_csv(PROCESSED / 'models_comparison.csv', index=False)


# ──────────────────────────────────────────────────────────────────────────
# 📊 Graphiques
# ──────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

targets = ["Temperature", "Precipitation"]

for i, target in enumerate(targets):

    data = df_comparison[df_comparison["Target"] == target]

    # RMSE
    axes[i, 0].bar(["Train", "Val"], [data["Train RMSE"].values[0], data["Val RMSE"].values[0]])
    axes[i, 0].set_title(f"{target} - RMSE")
    axes[i, 0].grid(alpha=0.3)

    # R²
    axes[i, 1].bar(["Train", "Val"], [data["Train R²"].values[0], data["Val R²"].values[0]])
    axes[i, 1].set_title(f"{target} - R²")
    axes[i, 1].grid(alpha=0.3)

plt.tight_layout()

plt.savefig(PROCESSED / "models_comparison.png", dpi=150)
plt.show()


print(f"\n💾 Graphiques sauvegardés : {PROCESSED / 'models_comparison.png'}")

📊 COMPARAISON DES MODÈLES

📋 TABLEAU COMPARATIF :

       Target  Modèle  Train RMSE   Val RMSE  Train R²   Val R²  Overfitting
  Temperature XGBoost    0.137537   0.295693  0.999711 0.998521    -0.158156
Precipitation XGBoost  172.354123 171.245853  0.648601 0.690507     1.108269

💾 Graphiques sauvegardés : processed/models_comparison.png


In [26]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 11 : SHAP Feature Importance
# ═══════════════════════════════════════════════════════════════════════════

import shap

print("=" * 80)
print("🔍 SHAP FEATURE IMPORTANCE")
print("=" * 80)

# ──────────────────────────────────────────────────────────────────────────
# SHAP pour XGBoost Température
# ──────────────────────────────────────────────────────────────────────────

print("\n🌡️ Calcul SHAP pour Température (XGBoost)...")

# Échantillonnage pour accélérer (SHAP peut être lent)
X_sample_temp = splits['temperature']['X_val'].sample(min(1000, len(splits['temperature']['X_val'])),
                                                       random_state=42)

explainer_temp = shap.TreeExplainer(xgb_temp)
shap_values_temp = explainer_temp.shap_values(X_sample_temp)

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_temp, X_sample_temp,
                  max_display=20, show=False)
plt.title('SHAP Feature Importance - Température', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(PROCESSED / 'shap_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance bar
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_temp, X_sample_temp,
                  plot_type="bar", max_display=20, show=False)
plt.title('SHAP Feature Importance (Bar) - Température', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(PROCESSED / 'shap_temperature_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# ──────────────────────────────────────────────────────────────────────────
# SHAP pour XGBoost Précipitation
# ──────────────────────────────────────────────────────────────────────────

print("\n🌧️ Calcul SHAP pour Précipitation (XGBoost)...")

X_sample_precip = splits['precipitation']['X_val'].sample(min(1000, len(splits['precipitation']['X_val'])),
                                                           random_state=42)

explainer_precip = shap.TreeExplainer(xgb_precip)
shap_values_precip = explainer_precip.shap_values(X_sample_precip)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_precip, X_sample_precip,
                  max_display=20, show=False)
plt.title('SHAP Feature Importance - Précipitation', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(PROCESSED / 'shap_precipitation.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_precip, X_sample_precip,
                  plot_type="bar", max_display=20, show=False)
plt.title('SHAP Feature Importance (Bar) - Précipitation', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(PROCESSED / 'shap_precipitation_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Analyses SHAP terminées et sauvegardées")

gc.collect()

🔍 SHAP FEATURE IMPORTANCE

🌡️ Calcul SHAP pour Température (XGBoost)...

🌧️ Calcul SHAP pour Précipitation (XGBoost)...

✅ Analyses SHAP terminées et sauvegardées


0

In [28]:
# ✅ Simplified training (XGBoost only + reduced data)

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd # Ensure pandas is imported

# 🔻 Reduce dataset size to avoid RAM crash
# Sample from the base_df, not the 'df' that might be an already sampled 'df_clean'
# Ensuring base_df is available, as in Cell 8
if 'engineered_datasets' in globals() and 'A_terrain_era5' in engineered_datasets:
    base_df = engineered_datasets['A_terrain_era5'].copy()
else:
    # Fallback: if engineered_datasets is not present, load from saved parquet
    print("Warning: 'engineered_datasets' not found. Loading 'engineered_A_terrain_era5.parquet' from disk.")
    base_df = pd.read_parquet(PROCESSED / 'engineered_A_terrain_era5.parquet')

df_small = base_df.sample(frac=0.3, random_state=42)

# Define target column explicitly. Using the temperature target from the overall notebook context.
target_col = 'hc_air_temperature_c'

if target_col not in df_small.columns:
    raise ValueError(f"Target column '{target_col}' not found in the sampled DataFrame.")

# 🔻 Reduce features (keep only first 30 numeric columns relevant for modeling)
# Exclude target, datetime, region_id, and year from features
numeric_cols = df_small.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    col for col in numeric_cols
    if col not in [target_col, 'year', 'datetime', 'region_id'] # datetime/region_id might be non-numeric but good to exclude
]
# Take the first 30 features from the filtered list
feature_cols = feature_cols[:30]

if not feature_cols:
    raise ValueError("No valid numeric features found after selection. Cannot train model.")

X = df_small[feature_cols].copy() # Features
y = df_small[target_col].copy()   # Target

# --- IMPORTANT: Handle NaNs in X and y ---
# 1. Filter out rows where the target (y) is NaN
initial_len = len(X)
valid_indices = y.notna()
X = X[valid_indices]
y = y[valid_indices]

if X.empty:
    raise ValueError("No valid data left after dropping NaN target values.")
print(f"Dropped {initial_len - len(X)} rows due to NaN target values.")

# 2. Fill any remaining NaNs in the feature set (X). Use 0 for tree models like XGBoost.
#    This needs to happen AFTER feature selection, but BEFORE the train/test split.
X.fillna(0, inplace=True)

# Ensure X and y are aligned after cleaning
if len(X) != len(y):
    raise ValueError("X and y lengths mismatch after NaN handling. This should not happen if valid_indices was applied correctly.")

# Split
# Check if X or y are empty after cleaning
if X.empty or y.empty:
    raise ValueError("X or y is empty after preprocessing. Cannot split data.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# 🚀 XGBoost (light config)
model = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.7,
    colsample_bytree=0.7,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"✅ XGBoost RMSE: {rmse:.3f}")


Dropped 290820 rows due to NaN target values.
X_train shape: (101837, 30), y_train shape: (101837,)
X_test shape: (25460, 30), y_test shape: (25460,)
✅ XGBoost RMSE: 3.560


In [54]:
import xgboost as xgb
from statsmodels.distributions.empirical_distribution import ECDF
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
import gc
import pickle
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path # Ensure Path is imported for PROCESSED

print("=" * 80)
print("📐 CALIBRATION MÉTÉOROLOGIQUE")
print("=" * 80)

# --- Vérification des dépendances ---
if 'splits' not in locals() and 'splits' not in globals():
    raise NameError("Variable 'splits' non définie. Exécutez d'abord la CELLULE 8.")

# Ensure PROCESSED is correctly set to the Google Drive path before loading models
# (It might have been redefined in prior cells like 9B)
if 'DRIVE_ROOT' in globals():
    PROCESSED = DRIVE_ROOT / 'data_processed_v3'
    PROCESSED.mkdir(exist_ok=True)
    print(f"🔧 PROCESSED path réinitialisé à: {PROCESSED}")
else:
    # Fallback if DRIVE_ROOT is not defined, although it should be by Cell 1
    PROCESSED = Path("processed") # This would use a local 'processed' directory
    PROCESSED.mkdir(exist_ok=True)
    print(f"⚠️  DRIVE_ROOT non trouvé. PROCESSED défini localement à: {PROCESSED}")


if 'df_main' not in locals() and 'df_main' not in globals():
    try:
        df_main = engineered_datasets['A_terrain_era5'].copy()
        print("🔧 df_main chargé depuis 'engineered_datasets'.")
    except (NameError, KeyError):
        print(f"📂 Chargement de 'engineered_A_terrain_era5.parquet' depuis {PROCESSED} sur le disque.")
        df_main = pd.read_parquet(PROCESSED / 'engineered_A_terrain_era5.parquet')
        df_main['datetime'] = pd.to_datetime(df_main['datetime'])

# --- Cartes des saisons (une seule définition) ---
season_map = {12: 0, 1: 0, 2: 0,   # DJF
               3: 1, 4: 1, 5: 1,   # MAM
               6: 2, 7: 2, 8: 2,   # JJA
               9: 3, 10: 3, 11: 3} # SON

# ----------------------------------------------------------------------
# Fonctions de calibration
# ----------------------------------------------------------------------
def bias_correction_by_region_month(y_true, y_pred, regions, months):
    """
    Calibration par biais (région x mois)
    """
    df_calib = pd.DataFrame({
        'true': y_true,
        'pred': y_pred,
        'region': regions,
        'month': months
    })
    bias_correction = df_calib.groupby(['region', 'month']).apply(
        lambda x: (x['true'] - x['pred']).mean()
    ).to_dict()
    y_pred_corrected = y_pred.copy()
    for (region, month), bias in bias_correction.items():
        mask = (regions == region) & (months == month)
        y_pred_corrected[mask] += bias
    return y_pred_corrected, bias_correction

def quantile_mapping_seasonal(y_true, y_pred, seasons):
    """
    Quantile mapping par saison (aligne la distribution prédite sur l'observée)
    """
    y_pred_mapped = y_pred.copy()
    for season_code in np.unique(seasons):
        mask = (seasons == season_code)
        y_true_season = y_true[mask]
        y_pred_season = y_pred[mask]
        if len(y_true_season) < 10 or len(y_pred_season) < 10:
            continue   # pas assez de données pour une calibration fiable

        # Quantiles de la distribution observée et prédite
        quantiles = np.linspace(0.01, 0.99, 99)
        q_true = np.quantile(y_true_season, quantiles)
        q_pred = np.quantile(y_pred_season, quantiles)

        # Supprimer les doublons pour interp1d (nécessite des x uniques et croissants)
        unique_idx = np.unique(q_pred, return_index=True)[1]
        q_pred_unique = q_pred[unique_idx]
        q_true_unique = q_true[unique_idx]

        if len(q_pred_unique) < 2:
            continue
        mapper = interp1d(q_pred_unique, q_true_unique,
                          bounds_error=False,
                          fill_value=(q_true_unique[0], q_true_unique[-1]))
        y_pred_mapped[mask] = mapper(y_pred_season)
    return y_pred_mapped

# ----------------------------------------------------------------------
# 1. CALIBRATION TEMPÉRATURE
# ----------------------------------------------------------------------
print("\n🌡️ Calibration Température...")

# Chargement du modèle XGBoost température
xgb_temp = xgb.Booster()
xgb_temp.load_model(PROCESSED / 'model_xgb_temperature.json')
print("🔧 xgb_temp chargé depuis le fichier.")

# Préparation des données
X_val_temp = splits['temperature']['X_val']
y_true_temp = splits['temperature']['y_val'].values

dmat_temp_val = xgb.DMatrix(X_val_temp.values)
y_pred_temp_raw = xgb_temp.predict(dmat_temp_val)

# Récupération des métadonnées (région, mois, saison) pour le jeu de validation
val_idx_temp = X_val_temp.index
val_info_temp = df_main.loc[val_idx_temp, ['region_id', 'month']].copy()
val_info_temp['season'] = val_info_temp['month'].map(season_map).astype('int8')

# Correction de biais
y_pred_temp_bias, bias_dict_temp = bias_correction_by_region_month(
    y_true_temp, y_pred_temp_raw,
    val_info_temp['region_id'].values,
    val_info_temp['month'].values
)

# Quantile mapping (après correction de biais)
y_pred_temp_calibrated = quantile_mapping_seasonal(
    y_true_temp, y_pred_temp_bias,
    val_info_temp['season'].values
)

# Métriques
rmse_raw = np.sqrt(mean_squared_error(y_true_temp, y_pred_temp_raw))
rmse_bias = np.sqrt(mean_squared_error(y_true_temp, y_pred_temp_bias))
rmse_calib = np.sqrt(mean_squared_error(y_true_temp, y_pred_temp_calibrated))

print(f"   RMSE brut           : {rmse_raw:.3f}°C")
print(f"   RMSE après biais    : {rmse_bias:.3f}°C (↓{100*(rmse_raw-rmse_bias)/rmse_raw:.1f}%)")
print(f"   RMSE après QM       : {rmse_calib:.3f}°C (↓{100*(rmse_raw-rmse_calib)/rmse_raw:.1f}%) ")

# Store residuals for uncertainty calculation
residuals_calib_t = y_true_temp - y_pred_temp_calibrated

# ----------------------------------------------------------------------
# 2. CALIBRATION PRÉCIPITATION
# ----------------------------------------------------------------------
print("\n🌧️ Calibration Précipitation...")

xgb_precip = xgb.Booster()
xgb_precip.load_model(PROCESSED / 'model_xgb_precipitation.json')
print("🔧 xgb_precip chargé depuis le fichier.")

X_val_precip = splits['precipitation']['X_val']
y_true_precip = splits['precipitation']['y_val'].values

dmat_precip_val = xgb.DMatrix(X_val_precip.values)
# Prédiction sur l'échelle log, puis exponentielle
y_pred_precip_raw = np.expm1(xgb_precip.predict(dmat_precip_val))

val_idx_precip = X_val_precip.index
val_info_precip = df_main.loc[val_idx_precip, ['region_id', 'month']].copy()
val_info_precip['season'] = val_info_precip['month'].map(season_map).astype('int8')

# Correction de biais
y_pred_precip_bias, bias_dict_precip = bias_correction_by_region_month(
    y_true_precip, y_pred_precip_raw,
    val_info_precip['region_id'].values,
    val_info_precip['month'].values
)
y_pred_precip_bias = np.clip(y_pred_precip_bias, 0, None)

# Quantile mapping
y_pred_precip_calibrated = quantile_mapping_seasonal(
    y_true_precip, y_pred_precip_bias,
    val_info_precip['season'].values
)
y_pred_precip_calibrated = np.clip(y_pred_precip_calibrated, 0, None)

rmse_raw_p = np.sqrt(mean_squared_error(y_true_precip, y_pred_precip_raw))
rmse_bias_p = np.sqrt(mean_squared_error(y_true_precip, y_pred_precip_bias))
rmse_calib_p = np.sqrt(mean_squared_error(y_true_precip, y_pred_precip_calibrated))

print(f"   RMSE brut           : {rmse_raw_p:.3f} mm")
print(f"   RMSE après biais    : {rmse_bias_p:.3f} mm (↓{100*(rmse_raw_p-rmse_bias_p)/rmse_raw_p:.1f}%) ")
print(f"   RMSE après QM       : {rmse_calib_p:.3f} mm (↓{100*(rmse_raw_p-rmse_calib_p)/rmse_raw_p:.1f}%) ")

# Store residuals for uncertainty calculation
residuals_calib_p = y_true_precip - y_pred_precip_calibrated

# ----------------------------------------------------------------------
# 3. VISUALISATIONS
# ----------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Température
axes[0,0].scatter(y_true_temp, y_pred_temp_raw, alpha=0.3, s=10, label='Brut')
axes[0,0].scatter(y_true_temp, y_pred_temp_calibrated, alpha=0.3, s=10, label='Calibré')
axes[0,0].plot([y_true_temp.min(), y_true_temp.max()],
               [y_true_temp.min(), y_true_temp.max()], 'r--', lw=2)
axes[0,0].set_xlabel('Température observée (°C)')
axes[0,0].set_ylabel('Température prédite (°C)')
axes[0,0].set_title('Température - Avant/Après Calibration')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

res_raw_t = y_true_temp - y_pred_temp_raw
res_cal_t = y_true_temp - y_pred_temp_calibrated # Use the correctly named variable
axes[0,1].hist([res_raw_t, res_cal_t], bins=50, alpha=0.6,
                label=['Brut', 'Calibré'])
axes[0,1].axvline(0, color='red', linestyle='--')
axes[0,1].set_xlabel('Résidus (°C)')
axes[0,1].set_title('Distribution des Résidus - Température')
axes[0,1].legend()

stats.probplot(res_cal_t, dist="norm", plot=axes[0,2])
axes[0,2].set_title('Q-Q Plot - Température (Calibré)')
axes[0,2].grid(alpha=0.3)

# Précipitation
axes[1,0].scatter(y_true_precip, y_pred_precip_raw, alpha=0.3, s=10, label='Brut')
axes[1,0].scatter(y_true_precip, y_pred_precip_calibrated, alpha=0.3, s=10, label='Calibré')
axes[1,0].plot([0, y_true_precip.max()], [0, y_true_precip.max()], 'r--', lw=2)
axes[1,0].set_xlabel('Précipitation observée (mm)')
axes[1,0].set_ylabel('Précipitation prédite (mm)')
axes[1,0].set_title('Précipitation - Avant/Après Calibration')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

res_raw_p = y_true_precip - y_pred_precip_raw
res_cal_p = y_true_precip - y_pred_precip_calibrated # Use the correctly named variable
axes[1,1].hist([res_raw_p, res_cal_p], bins=50, alpha=0.6,
                label=['Brut', 'Calibré'])
axes[1,1].axvline(0, color='red', linestyle='--')
axes[1,1].set_xlabel('Résidus (mm)')
axes[1,1].set_title('Distribution des Résidus - Précipitation')
axes[1,1].legend()

stats.probplot(res_cal_p, dist="norm", plot=axes[1,2])
axes[1,2].set_title('Q-Q Plot - Précipitation (Calibré)')
axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED / 'calibration_analysis.png', dpi=150)
plt.show()

# ----------------------------------------------------------------------
# 4. SAUVEGARDE
# ----------------------------------------------------------------------
calibration_functions = {
    'temperature': {
        'bias_correction': bias_dict_temp,
        'predictions_calibrated': y_pred_temp_calibrated # Save the correctly named variable
    },
    'precipitation': {
        'bias_correction': bias_dict_precip,
        'predictions_calibrated': y_pred_precip_calibrated # Save the correctly named variable
    }
}
with open(PROCESSED / 'calibration_functions.pkl', 'wb') as f:
    pickle.dump(calibration_functions, f)

print("\n✅ Calibration terminée et sauvegardée")
gc.collect()


📐 CALIBRATION MÉTÉOROLOGIQUE
🔧 PROCESSED path réinitialisé à: /content/drive/MyDrive/MyDrive/data_processed_v3

🌡️ Calibration Température...
🔧 xgb_temp chargé depuis le fichier.
   RMSE brut           : 0.296°C
   RMSE après biais    : 0.289°C (↓2.1%)
   RMSE après QM       : 0.498°C (↓-68.5%) 

🌧️ Calibration Précipitation...
🔧 xgb_precip chargé depuis le fichier.
   RMSE brut           : 173.991 mm
   RMSE après biais    : 169.261 mm (↓2.7%) 
   RMSE après QM       : 172.351 mm (↓0.9%) 

✅ Calibration terminée et sauvegardée


3029

### Inspection des valeurs manquantes dans X_train_precip

In [37]:
print(f"Shape de X_train_precip: {X_train_precip.shape}")
missing_values_xtrain_precip = X_train_precip.isnull().sum()
missing_values_xtrain_precip = missing_values_xtrain_precip[missing_values_xtrain_precip > 0]

if not missing_values_xtrain_precip.empty:
    print("Colonnes de X_train_precip avec des valeurs manquantes :")
    display(missing_values_xtrain_precip.sort_values(ascending=False))
    print(f"Total de valeurs NaN dans X_train_precip: {X_train_precip.isnull().sum().sum()}")
else:
    print("Aucune valeur manquante détectée dans X_train_precip.")

Shape de X_train_precip: (20799, 30)
Colonnes de X_train_precip avec des valeurs manquantes :


température__c            20799
température__1_c          20799
température__2_c          20799
hc_air_temperature_c_1    20799
hc_air_temperature_c_2    20799
soil_temperature_c        20723
température_sèche_c       20572
température_du_sol_c      20439
température_c             11504
hc_air_temperature_c       9299
temp_amplitude                4
temp_mean_sources             4
dtype: int64

Total de valeurs NaN dans X_train_precip: 186540


In [38]:
print("\n" + "="*80)
print("🔍 VÉRIFICATION DES VALEURS MANQUANTES DANS LES DATASETS INGÉNIERÉS")
print("="*80)

if 'engineered_datasets' not in globals():
    print("❌ Erreur : 'engineered_datasets' n'est pas défini. Veuillez exécuter les cellules précédentes.")
else:
    for name, df in engineered_datasets.items():
        print(f"\n--- Dataset : {name} ---")
        missing_values_count = df.isnull().sum()
        missing_values_percent = (df.isnull().sum() / len(df)) * 100

        missing_values_table = pd.DataFrame({
            'Missing Count': missing_values_count,
            'Missing Percentage': missing_values_percent
        })

        missing_values_table = missing_values_table[missing_values_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

        if not missing_values_table.empty:
            print("⚠️ Colonnes avec valeurs manquantes :")
            display(missing_values_table)
            print(f"Total NaN cells in {name}: {df.isnull().sum().sum()}")
        else:
            print(f"✅ Aucune valeur manquante détectée dans {name}.")

gc.collect()


🔍 VÉRIFICATION DES VALEURS MANQUANTES DANS LES DATASETS INGÉNIERÉS

--- Dataset : A_terrain_era5 ---
⚠️ Colonnes avec valeurs manquantes :


,Missing Count,Missing Percentage
battery_voltage_mv,1393724,100.000000
era5_precipitation,1393724,100.000000
vitesse_du_vent_2_m_s,1393724,100.000000
température_de_l'air__3_c,1393724,100.000000
era5_temp_min,1393724,100.000000
...,...,...
col_15,178545,12.810643
col_16,148868,10.681311
col_14,137860,9.891485
temp_mean_sources,57240,4.106982


Total NaN cells in A_terrain_era5: 109858968

--- Dataset : B_terrain_consensus ---
⚠️ Colonnes avec valeurs manquantes :


,Missing Count,Missing Percentage
era5_v_component_of_wind_10m,1393724,100.000000
era5_u_component_of_wind_10m,1393724,100.000000
era5_temperature_2m,1393724,100.000000
era5_temp_min,1393724,100.000000
latitude,1393724,100.000000
...,...,...
col_15,178545,12.810643
col_16,148868,10.681311
col_14,137860,9.891485
temp_mean_sources,57211,4.104902


Total NaN cells in B_terrain_consensus: 164447137

--- Dataset : C_terrain_all ---
⚠️ Colonnes avec valeurs manquantes :


,Missing Count,Missing Percentage
vitesse_du_vent_2_m_s,1393724,100.000000
battery_voltage_mv,1393724,100.000000
source,1393724,100.000000
latitude,1393724,100.000000
température_de_l'air__3_c,1393724,100.000000
...,...,...
col_15,178545,12.810643
col_16,148868,10.681311
col_14,137860,9.891485
temp_mean_sources,57211,4.104902


Total NaN cells in C_terrain_all: 150841317


19417

In [55]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 14 : Série Temporelle Réel vs Prédit
# ═══════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("📈 VISUALISATION SÉRIES TEMPORELLES")
print("=" * 80)

# Extract true values and their original dates from the splits dictionary
# These retain their original indices from df_main

y_true_temp = splits['temperature']['y_val'].values
val_indices_temp = splits['temperature']['y_val'].index
val_dates_temp = df_main.loc[val_indices_temp, 'datetime'].values

y_true_precip = splits['precipitation']['y_val'].values
val_indices_precip = splits['precipitation']['y_val'].index
val_dates_precip = df_main.loc[val_indices_precip, 'datetime'].values

# ═══════════════════════════════════════════════════════════════════════════
# Graphique interactif Plotly
# ═══════════════════════════════════════════════════════════════════════════

from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Température (°C)', 'Précipitation (mm)'),
    vertical_spacing=0.12
)

# Température
fig.add_trace(
    go.Scatter(x=val_dates_temp, y=y_true_temp, mode='lines',
               name='Observé', line=dict(color='blue', width=1)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=val_dates_temp, y=y_pred_temp_calibrated, mode='lines',
               name='Prédit (calibré)', line=dict(color='red', width=1, dash='dash')),
    row=1, col=1
)

# Précipitation
fig.add_trace(
    go.Scatter(x=val_dates_precip, y=y_true_precip, mode='lines',
               name='Observé', line=dict(color='blue', width=1), showlegend=False),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=val_dates_precip, y=y_pred_precip_calibrated, mode='lines',
               name='Prédit (calibré)', line=dict(color='red', width=1, dash='dash'), showlegend=False),
    row=2, col=1
)

fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Température (°C)", row=1, col=1)
fig.update_yaxes(title_text="Précipitation (mm)", row=2, col=1)

fig.update_layout(
    height=800,
    title_text="Séries Temporelles - Validation Set",
    hovermode='x unified'
)

fig.write_html(str(PROCESSED / 'timeseries_interactive.html'))
print(f"💾 Graphique interactif : {PROCESSED / 'timeseries_interactive.html'}")

fig.show()

# ═══════════════════════════════════════════════════════════════════════════
# Version statique matplotlib (zoom sur 1 mois)
# ═══════════════════════════════════════════════════════════════════════════

# Sélection d'un mois pour zoom
start_idx = 0
end_idx = min(720, len(val_dates_temp))  # ~30 jours si horaire

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Température
axes[0].plot(val_dates_temp[start_idx:end_idx], y_true_temp[start_idx:end_idx],
             label='Observé', linewidth=2, alpha=0.8)
axes[0].plot(val_dates_temp[start_idx:end_idx], y_pred_temp_calibrated[start_idx:end_idx],
             label='Prédit (calibré)', linewidth=2, alpha=0.8, linestyle='--')
axes[0].set_ylabel('Température (°C)')
axes[0].set_title('Série Temporelle - Température (Premier mois de validation)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Précipitation
axes[1].plot(val_dates_precip[start_idx:end_idx], y_true_precip[start_idx:end_idx],
             label='Observé', linewidth=2, alpha=0.8)
axes[1].plot(val_dates_precip[start_idx:end_idx], y_pred_precip_calibrated[start_idx:end_idx],
             label='Prédit (calibré)', linewidth=2, alpha=0.8, linestyle='--')
axes[1].set_ylabel('Précipitation (mm)')
axes[1].set_xlabel('Date')
axes[1].set_title('Série Temporelle - Précipitation (Premier mois de validation)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED / 'timeseries_monthly_zoom.png', dpi=150)
plt.show()

print("✅ Visualisations séries temporelles créées")

📈 VISUALISATION SÉRIES TEMPORELLES
💾 Graphique interactif : /content/drive/MyDrive/MyDrive/data_processed_v3/timeseries_interactive.html


✅ Visualisations séries temporelles créées


In [56]:
print("\n" + "="*80)
print("🔍 VÉRIFICATION DES VALEURS MANQUANTES")
print("="*80)

# Ensure df_main is defined - this assumes 'cleaned_datasets' is available from CELLULE 7
# If 'cleaned_datasets' is not defined, please run all cells from CELLULE 1 up to CELLULE 8.
if 'df_main' not in locals() and 'df_main' not in globals():
    try:
        df_main = cleaned_datasets['A_terrain_era5'].copy()
        print("🔧 df_main a été redéfini à partir de cleaned_datasets.")
    except NameError:
        print("❌ Erreur : 'cleaned_datasets' n'est pas défini. Veuillez exécuter toutes les cellules de 1 à 8.")
        raise # Re-raise if cleaned_datasets is missing


missing_values_count = df_main.isnull().sum()
missing_values_percent = (df_main.isnull().sum() / len(df_main)) * 100

missing_values_table = pd.DataFrame({
    'Missing Count': missing_values_count,
    'Missing Percentage': missing_values_percent
})

missing_values_table = missing_values_table[missing_values_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

if not missing_values_table.empty:
    print("\n⚠️ Colonnes avec valeurs manquantes après traitement :")
    display(missing_values_table)
    print(f"Total NaN cells in df_main: {df_main.isnull().sum().sum()}")
else:
    print("\n✅ Aucune valeur manquante détectée dans df_main.")

gc.collect()


🔍 VÉRIFICATION DES VALEURS MANQUANTES

⚠️ Colonnes avec valeurs manquantes après traitement :


,Missing Count,Missing Percentage
battery_voltage_mv,1393724,100.000000
era5_precipitation,1393724,100.000000
vitesse_du_vent_2_m_s,1393724,100.000000
température_de_l'air__3_c,1393724,100.000000
era5_temp_min,1393724,100.000000
...,...,...
col_15,178545,12.810643
col_16,148868,10.681311
col_14,137860,9.891485
temp_mean_sources,57240,4.106982


Total NaN cells in df_main: 109858968


358

In [57]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import gc

print("=" * 80)
print("🔮 FONCTION DE PRÉDICTION - NOUVELLE RÉGION")
print("=" * 80)

def predire_nouvelle_region(latitude: float,
                             longitude: float,
                             start_date: str,
                             end_date: str,
                             output_path: Optional[Path] = None) -> pd.DataFrame:
    """
    Prédit température et précipitation pour une nouvelle région

    Paramètres:
        latitude: Latitude de la région
        longitude: Longitude de la région
        start_date: Date de début (format 'YYYY-MM-DD')
        end_date: Date de fin (format 'YYYY-MM-DD')
        output_path: Chemin de sauvegarde CSV (optionnel)

    Returns:
        DataFrame avec prédictions horaires
    """

    print(f"\n📍 Prédiction pour nouvelle région :")
    print(f"   Lat/Lon : ({latitude}, {longitude})")
    print(f"   Période : {start_date} → {end_date}")

    # ═══════════════════════════════════════════════════════════════════════
    # 1. Extraction données satellite pour la région
    # ═══════════════════════════════════════════════════════════════════════

    print("\n📡 Extraction données satellite...")

    # SIMULATION (à remplacer par vraies API calls)
    dates = pd.date_range(start_date, end_date, freq='h')

    # Simulation données ERA5, Open-Meteo, NASA
    np.random.seed(42)
    df_new = pd.DataFrame({
        'datetime': dates,
        'latitude': latitude,
        'longitude': longitude,
        'era5_temperature': np.random.randn(len(dates)) * 5 + 20,
        'era5_precipitation': np.abs(np.random.randn(len(dates))) * 2,
        'openmeteo_temperature': np.random.randn(len(dates)) * 5 + 20,
        'nasa_temperature': np.random.randn(len(dates)) * 5 + 20,
        'era5_humidity': np.random.rand(len(dates)) * 40 + 40,
        'era5_wind_speed': np.abs(np.random.randn(len(dates))) * 3,
        'era5_radiation': np.abs(np.random.randn(len(dates))) * 200 + 100
    })

    print(f"   ✅ {len(df_new)} heures de données extraites")

    # ═══════════════════════════════════════════════════════════════════════
    # 2. Feature engineering (identique à l'entraînement)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n🔧 Feature engineering...")

    df_new['region_id'] = 'new_region'
    df_new = create_temporal_features(df_new)
    df_new = create_meteorological_features(df_new)
    df_new = create_lag_features(df_new, group_col='region_id')
    df_new = fill_missing_values_smart(df_new)

    # Conversion float32
    numeric_cols = df_new.select_dtypes(include=[np.number]).columns
    df_new[numeric_cols] = df_new[numeric_cols].astype('float32')

    # ═══════════════════════════════════════════════════════════════════════
    # 3. Préparation features (intersection avec features d'entraînement)
    # ═══════════════════════════════════════════════════════════════════════

    feature_cols_temp = splits['temperature']['feature_cols']
    feature_cols_precip = splits['precipitation']['feature_cols']

    # Features disponibles
    available_temp = [c for c in feature_cols_temp if c in df_new.columns]
    available_precip = [c for c in feature_cols_precip if c in df_new.columns]

    # Ajout colonnes manquantes avec 0
    for col in feature_cols_temp:
        if col not in df_new.columns:
            df_new[col] = 0

    for col in feature_cols_precip:
        if col not in df_new.columns:
            df_new[col] = 0

    X_new_temp = df_new[feature_cols_temp].copy()
    X_new_precip = df_new[feature_cols_precip].copy()

    # ═══════════════════════════════════════════════════════════════════════
    # 4. Prédictions
    # ═══════════════════════════════════════════════════════════════════════

    print("\n🤖 Prédictions...")

    # Température
    dmat_temp = xgb.DMatrix(X_new_temp)
    temp_pred_raw = xgb_temp.predict(dmat_temp)

    # Précipitation (inverse log)
    dmat_precip = xgb.DMatrix(X_new_precip)
    precip_pred_raw = np.expm1(xgb_precip.predict(dmat_precip))
    precip_pred_raw = np.clip(precip_pred_raw, 0, None)

    # ═══════════════════════════════════════════════════════════════════════
    # 5. Calibration (si région connue, sinon calibration moyenne)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n📐 Application calibration...")

    # Utilisation calibration moyenne (toutes régions/mois)
    # Pour région nouvelle, on applique la correction moyenne

    # Température : correction moyenne
    all_bias_temp = list(bias_dict_temp.values())
    mean_bias_temp = np.mean(all_bias_temp)
    temp_pred_calibrated = temp_pred_raw + mean_bias_temp

    # Précipitation : correction moyenne
    all_bias_precip = list(bias_dict_precip.values())
    mean_bias_precip = np.mean(all_bias_precip)
    precip_pred_calibrated = precip_pred_raw + mean_bias_precip
    precip_pred_calibrated = np.clip(precip_pred_calibrated, 0, None)

    # ═══════════════════════════════════════════════════════════════════════
    # 6. Incertitude (Bootstrap approximation simple)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n📊 Calcul intervalles de confiance...")

    # Approximation : ±1.96 * std des résidus de validation
    std_temp = np.std(residuals_calib_t)
    std_precip = np.std(residuals_calib_p)

    temp_lower = temp_pred_calibrated - 1.96 * std_temp
    temp_upper = temp_pred_calibrated + 1.96 * std_temp

    precip_lower = np.clip(precip_pred_calibrated - 1.96 * std_precip, 0, None)
    precip_upper = precip_pred_calibrated + 1.96 * std_precip

    # ═══════════════════════════════════════════════════════════════════════
    # 7. Construction DataFrame final
    # ═══════════════════════════════════════════════════════════════════════

    df_predictions = pd.DataFrame({
        'datetime': df_new['datetime'],
        'latitude': latitude,
        'longitude': longitude,
        'temperature_pred': temp_pred_calibrated,
        'temperature_lower_95': temp_lower,
        'temperature_upper_95': temp_upper,
        'precipitation_pred': precip_pred_calibrated,
        'precipitation_lower_95': precip_lower,
        'precipitation_upper_95': precip_upper
    })

    # ═══════════════════════════════════════════════════════════════════════
    # 8. Sauvegarde
    # ═══════════════════════════════════════════════════════════════════════

    if output_path is None:
        output_path = PROCESSED / f'predictions_{latitude}_{longitude}_{start_date}_{end_date}.csv'

    df_predictions.to_csv(output_path, index=False)

    print(f"\n✅ Prédictions sauvegardées : {output_path}")
    print(f"   {len(df_predictions)} heures prédites")
    print(f"\n📊 Statistiques :")
    print(f"   Température : {df_predictions['temperature_pred'].mean():.1f}°C ± {df_predictions['temperature_pred'].std():.1f}°C")
    print(f"   Précipitation : {df_predictions['precipitation_pred'].sum():.1f} mm (total)")

    return df_predictions


# ═══════════════════════════════════════════════════════════════════════════
# Test de la fonction
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TEST : Prédiction pour région exemple")
print("="*60)

# Exemple : Casablanca, Maroc (33.5731°N, 7.5898°W)
df_test_pred = predire_nouvelle_region(
    latitude=33.5731,
    longitude=-7.5898,
    start_date='2024-01-01',
    end_date='2024-01-31'
)

# Visualisation rapide
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Température
axes[0].plot(df_test_pred['datetime'], df_test_pred['temperature_pred'],
             label='Prédiction', linewidth=2)
axes[0].fill_between(df_test_pred['datetime'],
                      df_test_pred['temperature_lower_95'],
                      df_test_pred['temperature_upper_95'],
                      alpha=0.3, label='IC 95%')
axes[0].set_ylabel('Température (°C)')
axes[0].set_title('Prédictions Température - Nouvelle Région')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Précipitation
axes[1].plot(df_test_pred['datetime'], df_test_pred['precipitation_pred'],
             label='Prédiction', linewidth=2)
axes[1].fill_between(df_test_pred['datetime'],
                      df_test_pred['precipitation_lower_95'],
                      df_test_pred['precipitation_upper_95'],
                      alpha=0.3, label='IC 95%')
axes[1].set_ylabel('Précipitation (mm)')
axes[1].set_xlabel('Date')
axes[1].set_title('Prédictions Précipitation - Nouvelle Région')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED / 'prediction_new_region_example.png', dpi=150)
plt.show()

print("\n✅ Fonction de prédiction testée avec succès")


🔮 FONCTION DE PRÉDICTION - NOUVELLE RÉGION

TEST : Prédiction pour région exemple

📍 Prédiction pour nouvelle région :
   Lat/Lon : (33.5731, -7.5898)
   Période : 2024-01-01 → 2024-01-31

📡 Extraction données satellite...
   ✅ 721 heures de données extraites

🔧 Feature engineering...
   ✅ Features temporelles : 30 colonnes
   ✅ VPD calculé depuis era5_temperature et era5_humidity
   ✅ Features météo : 34 colonnes
   ✅ Lags temporels : 46 colonnes
   ✅ NaN remplis

🤖 Prédictions...

📐 Application calibration...

📊 Calcul intervalles de confiance...

✅ Prédictions sauvegardées : /content/drive/MyDrive/MyDrive/data_processed_v3/predictions_33.5731_-7.5898_2024-01-01_2024-01-31.csv
   721 heures prédites

📊 Statistiques :
   Température : 19.3°C ± 2.6°C
   Précipitation : 54452.7 mm (total)

✅ Fonction de prédiction testée avec succès
